<div style="display:flex; align-items:center; justify-content:center; gap:22px; flex-wrap:wrap; width:100%; max-width:980px; margin:0 auto 16px; padding:10px 16px; box-sizing:border-box; background:#ffffff; border:1px solid #e5e7eb; border-radius:8px; box-shadow:0 1px 2px rgba(0,0,0,0.04);">
  <span style="display:flex; align-items:center; justify-content:center; flex:1.25 1 220px; min-width:180px; max-width:300px; height:76px;">
    <img src="assets/images/logos/nvidia-logo.png" alt="NVIDIA" style="display:block; max-width:100%; max-height:62px; width:auto; height:auto; object-fit:contain;">
  </span>
  <span style="display:flex; align-items:center; justify-content:center; flex:1 1 130px; min-width:110px; max-width:180px; height:58px;">
    <img src="assets/images/logos/eneos-orange.png" alt="ENEOS" style="display:block; max-width:100%; max-height:30px; width:auto; height:auto; object-fit:contain;">
  </span>
  <span style="display:flex; align-items:center; justify-content:center; flex:1.45 1 260px; min-width:220px; max-width:360px; height:86px;">
    <img src="assets/images/logos/matlantis.png" alt="Matlantis" style="display:block; max-width:100%; max-height:82px; width:auto; height:auto; object-fit:contain;">
  </span>
  <span style="display:flex; align-items:center; justify-content:center; flex:1 1 120px; min-width:100px; max-width:160px; height:58px;">
    <img src="assets/images/logos/ovito_logo.png" alt="OVITO" style="display:block; max-width:100%; max-height:32px; width:auto; height:auto; object-fit:contain;">
  </span>
</div>

# Batched Atomistic Simulation with <span style="color:#76b900; font-weight:600;">NVIDIA ALCHEMI</span>

<p style="margin-top:0; color:#555; font-size:0.95em;">
Inspiration for this tutorial is drawn from a fruitful three-way collaboration between NVIDIA ALCHEMI, ENEOS, and Matlantis: NVIDIA ALCHEMI provides the accelerated tutorial/runtime layer integrated into the Matlantis computational platform; ENEOS leverages Matlantis to study production oxygen evolution reaction (OER) catalysis search across a million-candidate space. OVITO is used for structure inspection and rendering. The notebook below is a simplified adsorption tutorial inspired by that real production search.
</p>

🔗 **ALCHEMI resources:** [ALCHEMI hub](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) · [Toolkit GitHub](https://github.com/NVIDIA/nvalchemi-toolkit) · [Toolkit-Ops GitHub](https://github.com/NVIDIA/nvalchemi-toolkit-ops)

📰 **ALCHEMI blogs:** [ALCHEMI discovery blog](https://developer.nvidia.com/blog/revolutionizing-ai-driven-material-discovery-using-nvidia-alchemi/) · [Toolkit intro blog](https://developer.nvidia.com/blog/building-custom-atomistic-simulation-workflows-for-chemistry-and-materials-science-with-nvidia-alchemi-toolkit/) · [Toolkit-Ops blog](https://developer.nvidia.com/blog/accelerating-ai-powered-chemistry-and-materials-science-simulations-with-nvidia-alchemi-toolkit-ops/)

![Adsorption at scale with NVIDIA ALCHEMI Toolkit](assets/images/banner_adsorption_scale_alchemi.jpg)

<!-- **Runtime target:** ALCHEMI Toolkit on a CUDA GPU  
**Teaching path:** batched MLIP relaxation with explicit structure, energy, and validation checkpoints  
**Companion check:** [OC20Dense accuracy and reproducibility](oc20dense-accuracy-reproducibility-check.ipynb) -->

*This tutorial uses adsorption configuration search as an example of a broader shift: batched, GPU-native atomistic simulation lets a researcher run throughput calculations on a single modern GPU and route better-ranked structures into production workflows or whatever downstream review or validation path the project requires.*

**What you will do**

1. Use a small H<sub>2</sub>O example to see how batching changes runtime.
2. Run a limited OC20Dense reproducibility and model-sanity check against released DFT trajectories.
3. Build chemically meaningful adsorption starting structures.
4. Relax, rank, visualize, and interpret the final adsorption geometries.

## Chemical discovery starts as a search problem

Chemical discovery begins with an enormous, almost limitless solution space. A research goal can branch across composition, molecular structure, crystal structure, environment, target property, and practical constraints such as cost, safety, synthesis route, or lifetime. The first step is therefore not to run a model; those come soon. It is to define a meaningful search space.

A useful computational workflow narrows that space in stages:

- **Frame the scientific question.** Decide what is being optimized or rationalized: stability, selectivity, conductivity, degradation, binding, diffusion, emission, or another property.
- **Apply chemical judgement.** Use known chemistry, physics, synthesis constraints, and domain knowledge (human or even agentic) to remove candidates that are irrelevant or impossible for the intended application.
<div style="margin:18px auto 20px; max-width:920px; text-align:center; font-family:Inter, Arial, sans-serif;">
  <div style="margin:0 auto 7px; width:100%; padding:12px 18px; box-sizing:border-box; border:2px solid #d1d5db; background:#f9fafb; color:#1f2937; border-radius:8px;">
    <div style="font-weight:700; font-size:1.02em;">Chemical solution space</div>
    <div style="font-size:0.9em;">composition, structure, environment, target property, practical constraints</div>
  </div>
  <div style="margin:0 auto 7px; width:88%; padding:11px 18px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#ffffff; color:#1f2937; border-radius:8px;">
    <div style="font-weight:700;">Frame the scientific question</div>
    <div style="font-size:0.9em;">stability, selectivity, binding, diffusion, degradation, conductivity, emission</div>
  </div>
  <div style="margin:0 auto 7px; width:76%; padding:11px 18px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#ffffff; color:#1f2937; border-radius:8px;">
    <div style="font-weight:700;">Apply chemical judgement</div>
    <div style="font-size:0.9em;">remove irrelevant, impossible, unsafe, or out-of-scope candidates</div>
  </div>
  <div style="margin:0 auto 7px; width:64%; padding:11px 18px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#ffffff; color:#1f2937; border-radius:8px;">
    <div style="font-weight:700;">Define the candidate set</div>
    <div style="font-size:0.9em;">molecules, conformers, crystals, defects, interfaces, adsorption geometries</div>
  </div>
  <div style="margin:0 auto 7px; width:52%; padding:11px 18px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#f9fafb; color:#1f2937; border-radius:8px;">
    <div style="font-weight:700;">Choose the evidence ladder</div>
    <div style="font-size:0.9em;">screening signal, validation path, metadata</div>
  </div>
  <div style="margin:0 auto 7px; width:42%; padding:11px 18px; box-sizing:border-box; border:2px solid #76b900; background:#edf7e2; color:#18230f; border-radius:8px;">
    <div style="font-weight:700;">Batched GPU evaluation</div>
    <div style="font-size:0.9em;">evaluate many plausible structures in one GPU workflow</div>
  </div>
  <div style="margin:0 auto; width:32%; min-width:250px; padding:11px 18px; box-sizing:border-box; border:2px solid #d1d5db; background:#f9fafb; color:#1f2937; border-radius:8px;">
    <div style="font-weight:700;">Ranked candidates</div>
    <div style="font-size:0.9em;">inspect, compare, and escalate the strongest cases</div>
  </div>
</div>

Even after this narrowing, the candidate set can still contain hundreds, thousands, or millions of structures. The practical bottleneck is no longer only the model; it is the ability to evaluate many reasonable structures without turning every idea into a separate campaign.

This is where batching changes the practical landscape. Instead of turning every exploratory idea into a separate calculation campaign, large sets of MLIP relaxations can now run as throughput calculations on a single modern GPU.

Surface adsorption gives this general search problem a concrete shape. Once a molecule-on-surface question is selected, the workflow still has to choose the slab model: composition, Miller index, termination, supercell, coverage, frozen-layer convention, and surface preparation. Only then does the local configuration search begin: adsorption sites, molecular orientation, height, and sometimes multiple starting geometries for the same chemical idea. Adsorption is part of many real discovery pipelines: catalysis, separations, water harvesting, OER materials, framework materials, and surface chemistry. ALCHEMI enters at this point to help researchers navigate solution spaces of interest with GPU-native batched workflows.


## Where NVIDIA ALCHEMI fits

[<span style="color:#76b900; font-weight:600;">NVIDIA ALCHEMI</span>](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) - the AI Lab for Chemistry and Materials Innovation - is built around a clear mission: accelerate chemical and materials discovery with AI. In practice, ALCHEMI brings together delivery modes such as domain-specific [NIM microservices](https://docs.nvidia.com/nim/alchemi/alchemi-bgr/latest/index.html), [ALCHEMI Toolkit](https://nvidia.github.io/nvalchemi-toolkit/) building blocks, and [Toolkit-Ops](https://nvidia.github.io/nvalchemi-toolkit-ops/) kernels for AI-enabled atomistic simulation. This tutorial focuses on the direct Toolkit path: use familiar Python structure tools, convert structures into Toolkit data objects, and run batched GPU geometry relaxations that are easy to inspect and extend.

ALCHEMI provides Toolkit data structures, optimizers, simulation components, and GPU kernels that can connect to familiar, established structure pipelines. In this notebook, the practical value is **batched**, **customizable**, **GPU-native** execution: familiar structure objects in, batched relaxation and simulation out.

The ALCHEMI pieces used in this tutorial are:

- <strong>Toolkit</strong> **Direct Python workflow.** This notebook uses the [ALCHEMI Toolkit API](https://nvidia.github.io/nvalchemi-toolkit/): build structures with ASE/pymatgen, convert them into `AtomicData`, pack them into a `Batch`, run a model and optimizer, then inspect the relaxed structures and metadata in Python.
- <span style="color:#00a3e0"><strong>Toolkit-Ops</strong></span> **Accelerated operations under the hood.** [Toolkit-Ops](https://github.com/NVIDIA/nvalchemi-toolkit-ops) is the GPU-kernel layer for common batched atomistic operations such as neighbor lists, DFT-D3 dispersion corrections, and long-range electrostatics. The tutorial does not ask readers to call Toolkit-Ops directly, but it is part of the ALCHEMI stack that makes high-throughput simulation practical.
- **Pluggable MLIPs.** The runnable tutorial path uses MACE-MP/MACE-MPA checkpoints: `medium-mpa-0` for the main adsorption screen, with other open MACE-MP sizes used to compare throughput and memory use. Other MACE models, including MACE-MH-1 with an OC20 surface head, can be tested separately by changing the model setup and following the applicable upstream license. In other ALCHEMI workflows, the model can change with the chemistry while the batching, metadata, and verification pattern remains the same.

The discovery loop is the complete cycle: define a candidate set, generate valid structures with established tools, run batched GPU relaxations or simulations, rank reliable outputs, inspect failures, and decide which cases are ready for the next project-specific review step.

![ALCHEMI Toolkit and Toolkit-Ops architecture](assets/images/alchemi_toolkit_architecture.png)

## Tools at a glance

Before the code starts, here is the role of each scientific tool in this workflow. The stack is intentionally interoperable: use familiar structure tools, accelerate the bottleneck, and keep the outputs inspectable.

| Tool | What it does here |
|---|---|
| **NVIDIA GPU/CUDA** | Supplies the accelerated execution layer that makes many independent relaxations practical as a throughput calculation on one GPU. |
| **ALCHEMI Toolkit** | Connects atomistic structures to batched GPU execution, model wrappers, optimizers, constraints, and workflow metadata. |
| **FIRE2** | Geometry-optimization algorithm (*Fast Inertial Relaxation Engine*, version 2) provided by the Toolkit. It iteratively moves atoms downhill in energy until forces drop below a target threshold. |
| **MACE MLIPs** | Evaluate energies and forces for periodic slab and molecule calculations. The runnable adsorption screen uses open MACE checkpoints, with `medium-mpa-0` as the default and small-vs-large open MACE sizes used for the batch-size trade-off exercise. |
| **DFT** | *Density Functional Theory*: the quantum-chemistry reference method that MACE is compared against. The tutorial reports MACE results next to DFT-computed reference adsorption energies wherever available. |
| **D3(BJ)** | Grimme D3 dispersion correction with Becke-Johnson damping; a van der Waals add-on to DFT. Kept disabled in this tutorial to match the OC20 reference convention. |
| **AdsorbML-style search** | Provides the workflow pattern: generate many plausible adsorbate placements, relax them, rank the final structures, and inspect failures instead of trusting one initial guess. |
| **OC20Dense** | A densely sampled subset of the Open Catalyst 2020 dataset; provides DFT-relaxed structures and DFT-computed adsorption energies used for the closed-shell H2O/NH3/N2 accuracy validation in the companion notebook. |
| **ASE** | Builds and stores atomistic structures in Python: gas molecules, slabs, adsorbate-slab starts, and relaxed outputs. |
| **pymatgen** | Builds crystal surfaces and terminations from bulk crystal structures, especially for oxide slabs where the surface model matters. |
| **OVITO** | Scientific molecular rendering platform. |


Surface adsorption makes the search problem easy to see. A molecule can approach the same surface through different atoms, different orientations, and different sites. Several starts may look chemically reasonable, but after relaxation they can settle into different local minima.

The 2023 AdsorbML study showed a practical shift for adsorption-energy workflows: instead of relaxing one manually chosen pose, generate many plausible adsorbate-surface configurations, use ML potentials to relax or filter them, and reserve higher-fidelity effort for the resulting shortlist. One balanced setting reported finding the lowest-energy configuration 87.36% of the time with about a 2000x speedup ([Lan et al., 2023](https://doi.org/10.1038/s41524-023-01121-5)).

This tutorial borrows that workflow logic. After the surface model is chosen, the calculation enumerates a local adsorption parameter space: candidate sites, orientations, heights, and related starting geometries. The goal is not to trust one plausible initial guess, but to search the defined configuration space and identify the lowest-energy relaxed structure found within it.

A batched search therefore asks a more useful question:

> **Given this surface model and adsorption parameter space, what is the best relaxed structure the workflow can find?**

ALCHEMI makes that search practical by running many independent starting structures as a batched GPU workflow, so the parameter sweep becomes part of the calculation rather than an informal manual preselection.

The same logic appears across many chemical domains. For adsorption it is sites and orientations. For molecules it may be conformers or protonation states. For materials it may be defects, terminations, dopants, interfaces, or local atomic arrangements. In catalysis, separations, water harvesting, OER frameworks, and surface chemistry, batching makes it practical to test many candidates instead of treating one starting structure as an invisible assumption.


## Roadmap: batching first, model check before screening

Batched relaxation only helps if the model is suitable for the chemistry being searched. Here that means an MLIP that has seen surface chemistry and can handle adsorbate-slab geometries well enough to support screening.

The notebook therefore builds the story in order: first make the Toolkit batching pattern concrete with a small H<sub>2</sub>O example, then check the selected open MACE path against released OC20Dense adsorption data, and only then use the same workflow for a broader adsorption search.

The opening workflow is:

- introduce the ALCHEMI Toolkit objects with a simple molecule batch;
- measure what batching changes computationally;
- check energy and relaxation behavior on known DFT trajectory data;
- generate a batched adsorption search for the systems we want to explore;
- relax, rank, and inspect the resulting geometries.

The validation checkpoint is part of the tutorial flow, not a separate promise: the OC20Dense-backed cells use exact starting structures, DFT-relaxed final structures, DFT-level adsorption-energy targets, clean-slab references, gas references, MACE single-point energies, and Toolkit relaxations that can be audited from dataset-backed files. The companion [OC20Dense accuracy and reproducibility notebook](oc20dense-accuracy-reproducibility-check.ipynb) keeps the deeper audit in one place: source-structure reruns, DFT trajectory arithmetic, MACE adsorption-energy subtraction, parity plots, and OVITO structure checks.

The generated adsorption panel is a screening example. It shows how to build, relax, rank, and inspect many candidate geometries. Quantitative comparison to literature requires matched surface models, coverage, functional, dispersion treatment, constraints, and adsorption-energy convention.

---

## Control panel

Use this cell to choose how the notebook will run. Later cells read these same settings, so one edit here changes the whole workflow.

The choices below are plain Python variables. Edit the values directly in the next cell.

There are two compute scopes:

- `RUN_SCOPE = "short"` runs one representative adsorption example with six starting structures. Use it to check the workflow, kernel, GPU connection, and result tables quickly.
- `RUN_SCOPE = "full"` runs the complete active adsorption grid defined in this notebook. Use it when you want the full site and orientation search for every adsorbate-surface example.

There are also two result sources:

- `"compute"` reruns the section and writes a new timestamped output folder.
- `"saved"` shows existing results, useful for reading the tutorial without waiting for GPU work.

Leave `SAVED_TUTORIAL_RUN_ID` and `SAVED_ACCURACY_RUN_ID` as `None` unless you want to reopen a specific previous run. Use `"latest-complete"` only when you want the newest live run that passes completeness checks.

Keep `REFRESH_SAVED_RESULTS = False` unless you are deliberately rebuilding the shipped saved results.

This version is Toolkit-only: relaxations run through the ALCHEMI Toolkit on the cluster GPU.

D3 is available in Toolkit workflows, but it is disabled here because the OC20Dense validation data follows the non-D3 OC20 convention. If you enable D3 for another dataset or application, record the explicit parameters in the run metadata.


In [ ]:
import os
import sys
from datetime import datetime

# === Run choices ===========================================================

# "short" checks the workflow with one small example.
# "full" runs the complete active grid defined in this notebook.
RUN_SCOPE = "full"

VALID_RUN_SCOPES = {"short", "full"}
if RUN_SCOPE not in VALID_RUN_SCOPES:
    raise ValueError(f"RUN_SCOPE must be one of {sorted(VALID_RUN_SCOPES)}, got {RUN_SCOPE!r}")
RUN_SCOPE_LABEL = {
    "short": "short check",
    "full": "full grid",
}[RUN_SCOPE]

# Choose whether each section recomputes results or reads saved outputs.
# "compute" writes to a timestamped live run; "saved" reads existing results
# unless a SAVED_*_RUN_ID below selects a previous live run.
TUTORIAL_RESULT_SOURCE = "compute"    # "compute" or "saved"
VALIDATION_RESULT_SOURCE = "compute"  # "compute" or "saved"

VALID_RESULT_SOURCES = {"compute", "saved"}
for name, value in {
    "TUTORIAL_RESULT_SOURCE": TUTORIAL_RESULT_SOURCE,
    "VALIDATION_RESULT_SOURCE": VALIDATION_RESULT_SOURCE,
}.items():
    if value not in VALID_RESULT_SOURCES:
        raise ValueError(f"{name} must be one of {sorted(VALID_RESULT_SOURCES)}, got {value!r}")

# Derived flag for tutorial output routing. Validation uses
# VALIDATION_RESULT_SOURCE directly so the validation switch stays readable.
USE_SAVED_TUTORIAL_RESULTS = TUTORIAL_RESULT_SOURCE == "saved"

# Leave these as None for the shipped saved results. Set one explicit timestamp
# to reopen a previous live run, for example "20260518-143022".
# Set "latest-complete" only when you want the newest live run that passes
# the required-file checks for this scope.
SAVED_TUTORIAL_RUN_ID = None
SAVED_ACCURACY_RUN_ID = None

# Intentional overwrite switch for shipped saved results.
# Leave False for reading, presenting, and exploratory runs. Set True only when
# you mean to rebuild the saved tutorial and validation outputs.
REFRESH_SAVED_RESULTS = False

LIVE_RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
if REFRESH_SAVED_RESULTS:
    TUTORIAL_RESULT_SOURCE = "compute"
    VALIDATION_RESULT_SOURCE = "compute"
    USE_SAVED_TUTORIAL_RESULTS = False
    SAVED_TUTORIAL_RUN_ID = None
    SAVED_ACCURACY_RUN_ID = None

# The computational story does not depend on local rendering. Leave this False
# on a headless workstation; set True when VisRTX is available and you want
# result structure images regenerated inside the notebook.
REQUIRE_VISRTX_RENDER = False

# Label used in result tables and run metadata.
EXECUTION_PATH = "toolkit"


In [ ]:
# Prepare the notebook workspace, output folders, and package caches.
# Start the kernel from this tutorial folder or from the repository root.
# === Notebook housekeeping =================================================
import importlib
import os
import sys
from pathlib import Path

tutorial_folder = Path.cwd().resolve()
if not (tutorial_folder / "helpers" / "__init__.py").exists():
    candidate = tutorial_folder / "part-1-batched-adsorption"
    if (candidate / "helpers" / "__init__.py").exists():
        tutorial_folder = candidate.resolve()
    else:
        raise RuntimeError(
            "Start Jupyter from this tutorial folder or from the repository root."
        )

TUTORIAL_ROOT = tutorial_folder
os.chdir(TUTORIAL_ROOT)

clean_sys_path = []
for entry in sys.path:
    if not entry:
        continue
    try:
        resolved = Path(entry).resolve()
    except (OSError, RuntimeError):
        clean_sys_path.append(entry)
        continue
    if resolved == TUTORIAL_ROOT:
        continue
    if (resolved / "alchemi-mace-adsorption-search.ipynb").exists():
        continue
    clean_sys_path.append(entry)
sys.path[:] = [str(TUTORIAL_ROOT), *clean_sys_path]

# During notebook development, refresh helper modules after file edits before importing them.
ipython = globals().get("get_ipython", lambda: None)()
if ipython is not None:
    try:
        ipython.run_line_magic("reload_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")
    except Exception:
        pass

for module_name in [name for name in list(sys.modules) if name == "helpers" or name.startswith("helpers.")]:
    del sys.modules[module_name]
importlib.invalidate_caches()

from helpers import (
    list_live_runs,
    make_tutorial_relpath,
    resolve_run_roots,
    write_artifact_index,
    write_run_manifest,
)

tutorial_relpath = make_tutorial_relpath(TUTORIAL_ROOT)

RUN_ROOTS = resolve_run_roots(
    tutorial_root=TUTORIAL_ROOT,
    run_scope=RUN_SCOPE,
    use_saved_tutorial_results=USE_SAVED_TUTORIAL_RESULTS,
    use_saved_accuracy_results=(VALIDATION_RESULT_SOURCE == "saved"),
    saved_tutorial_run_id=SAVED_TUTORIAL_RUN_ID,
    saved_accuracy_run_id=SAVED_ACCURACY_RUN_ID,
    refresh_saved_results=REFRESH_SAVED_RESULTS,
    live_run_id=LIVE_RUN_ID,
)

# === Local paths ===========================================================
PRECOMPUTED_OUTPUT_DIR = "outputs/precomputed"
PRECOMPUTED_TUTORIAL_OUTPUT_DIR = RUN_ROOTS.official_tutorial_root.as_posix()
PRECOMPUTED_ACCURACY_OUTPUT_DIR = RUN_ROOTS.official_accuracy_root.as_posix()
LIVE_OUTPUT_DIR = RUN_ROOTS.live_root.as_posix()
TUTORIAL_OUTPUT_DIR = RUN_ROOTS.tutorial_output_dir.as_posix()
ACCURACY_OUTPUT_DIR = RUN_ROOTS.accuracy_output_dir.as_posix()
CACHE_DIR = RUN_ROOTS.cache_dir.as_posix()
PLOTS_DIR = RUN_ROOTS.plots_dir.as_posix()
RUNTIME_CACHE_DIR = RUN_ROOTS.runtime_cache_dir.as_posix()
OUTPUT_DIR = TUTORIAL_OUTPUT_DIR
SURFACE_SCREEN_OUTPUT_ROOT = RUN_ROOTS.surface_screen_root.as_posix()

ASSETS_DIR = "assets"
IMAGES_DIR = os.path.join(ASSETS_DIR, "images")
PRESENTATION_PLOTS_DIR = os.path.join(PRECOMPUTED_TUTORIAL_OUTPUT_DIR, "plots")

# Keep model and library caches beside the notebook so setup is repeatable.
os.environ["XDG_CACHE_HOME"] = str(TUTORIAL_ROOT / RUNTIME_CACHE_DIR / "xdg")
os.environ["TORCH_HOME"] = str(TUTORIAL_ROOT / RUNTIME_CACHE_DIR / "torch")
os.environ["HF_HOME"] = str(TUTORIAL_ROOT / RUNTIME_CACHE_DIR / "hf")
os.environ["WARP_CACHE_PATH"] = str(TUTORIAL_ROOT / RUNTIME_CACHE_DIR / "warp")
os.environ["REFRESH_SAVED_RESULTS"] = "1" if REFRESH_SAVED_RESULTS else "0"
_OVERWRITE_FLAG = "1" if REFRESH_SAVED_RESULTS else "0"
os.environ["ALCHEMI_ALLOW_ARTIFACT_OVERWRITE"] = _OVERWRITE_FLAG
# Live compute runs can be repeated in the same timestamped folder. Shipped
# saved results remain protected unless REFRESH_SAVED_RESULTS is explicit.
os.environ["ALCHEMI_ALLOW_CACHE_OVERWRITE"] = "1" if not USE_SAVED_TUTORIAL_RESULTS else _OVERWRITE_FLAG

print("Tutorial folder                  : .")
print(f"Tutorial output source            : {RUN_ROOTS.tutorial_source_label}")
print(f"Accuracy output source            : {RUN_ROOTS.accuracy_source_label}")
print(f"Live run root                     : {tutorial_relpath(LIVE_OUTPUT_DIR)}")
recent_live_runs = list_live_runs(TUTORIAL_ROOT, run_scope=RUN_SCOPE)
if recent_live_runs:
    print("Recent live runs:")
    for row in recent_live_runs[:5]:
        print(
            f"  {row['run_id']} | scope={row['run_scope']} | "
            f"tutorial_complete={row['tutorial_complete']} | "
            f"accuracy_complete={row['accuracy_complete']}"
        )


In [ ]:
# Runtime settings shared by the live Toolkit cells.
# === Notebook run settings =================================================
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"

# Acceleration settings. cuEquivariance stays enabled because it is useful for this
# GPU tutorial path. torch.compile is opt-in here: this notebook intentionally
# changes batch sizes and structure shapes, which can trigger repeated Dynamo
# recompilation warnings and warm-up overhead. Turn it on for repeated fixed-shape
# production runs after measuring that it helps on your GPU/runtime.
TOOLKIT_COMPILE_MODEL = False  # set True only for fixed-shape runs after checking compile warm-up/recompile behavior
TOOLKIT_ENABLE_CUEQ = True     # use cuEquivariance kernels when the model/runtime supports them
TOOLKIT_MATMUL_PRECISION = "high"  # enables TF32 matmul on NVIDIA GPUs; use "highest" for strict FP32 reruns

print(f"RUN_SCOPE                         : {RUN_SCOPE} ({RUN_SCOPE_LABEL})")
print(f"EXECUTION_PATH                     : {EXECUTION_PATH}")
print(f"TUTORIAL_RESULT_SOURCE             : {TUTORIAL_RESULT_SOURCE}")
print(f"VALIDATION_RESULT_SOURCE           : {VALIDATION_RESULT_SOURCE}")
print(f"REFRESH_SAVED_RESULTS              : {REFRESH_SAVED_RESULTS}")
print(f"SAVED_TUTORIAL_RUN_ID              : {SAVED_TUTORIAL_RUN_ID or 'official'}")
print(f"SAVED_ACCURACY_RUN_ID              : {SAVED_ACCURACY_RUN_ID or 'official'}")
print(f"TUTORIAL_OUTPUT_DIR                : {TUTORIAL_OUTPUT_DIR}")
print(f"ACCURACY_OUTPUT_DIR                : {ACCURACY_OUTPUT_DIR}")
print(f"CACHE_DIR                          : {CACHE_DIR}")
print(f"SURFACE_SCREEN_OUTPUT_ROOT         : {SURFACE_SCREEN_OUTPUT_ROOT}")
print(f"TOOLKIT_DEVICE                     : {TOOLKIT_DEVICE}")
print(f"TOOLKIT_DTYPE                      : {TOOLKIT_DTYPE}")
print(f"TOOLKIT_COMPILE_MODEL              : {TOOLKIT_COMPILE_MODEL}")
print(f"TOOLKIT_ENABLE_CUEQ                : {TOOLKIT_ENABLE_CUEQ}")
print(f"TOOLKIT_MATMUL_PRECISION           : {TOOLKIT_MATMUL_PRECISION}")
print(f"REQUIRE_VISRTX_RENDER              : {REQUIRE_VISRTX_RENDER}")


## Package versions and imports

In [ ]:
# Bootstrap public Python dependencies inside the active Jupyter kernel when the
# remote runtime image is fixed or cannot be rebuilt. This cell intentionally
# handles public notebook packages only; the ALCHEMI Toolkit remains checked in
# the Toolkit preflight because it may require the configured NVIDIA/private index.
import importlib
import importlib.util
import subprocess
import sys

INSTALL_MISSING_KERNEL_PACKAGES = True

# Known-good versions for this tutorial runtime. The cell installs only packages
# that are missing from the active kernel; it does not upgrade packages that are
# already importable.
KERNEL_PUBLIC_DEPENDENCIES = {
    "ase": "ase==3.28.0",
    "numpy": "numpy==2.4.4",
    "pandas": "pandas==2.3.3",
    "matplotlib": "matplotlib==3.10.9",
    "pymatgen": "pymatgen==2026.5.4",
    "pydantic": "pydantic==2.13.4",
    "ipywidgets": "ipywidgets==8.1.8",
    "tqdm": "tqdm",
    "ovito": "ovito==3.15.0",
}

missing_imports = [
    import_name
    for import_name in KERNEL_PUBLIC_DEPENDENCIES
    if importlib.util.find_spec(import_name) is None
]

if missing_imports and not INSTALL_MISSING_KERNEL_PACKAGES:
    packages = [KERNEL_PUBLIC_DEPENDENCIES[name] for name in missing_imports]
    raise RuntimeError(
        "Missing notebook packages: "
        + ", ".join(missing_imports)
        + ". To install them into this kernel, set "
        + "INSTALL_MISSING_KERNEL_PACKAGES = True and rerun this cell. Equivalent command: "
        + f"{sys.executable} -m pip install "
        + " ".join(packages)
    )

if missing_imports:
    packages = [KERNEL_PUBLIC_DEPENDENCIES[name] for name in missing_imports]
    print("Installing missing public notebook packages into this kernel:", ", ".join(packages))
    try:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                *packages,
            ]
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Kernel package install failed. Check network/index access, then run manually: "
            + f"{sys.executable} -m pip install "
            + " ".join(packages)
        ) from exc
    importlib.invalidate_caches()

still_missing = [
    import_name
    for import_name in KERNEL_PUBLIC_DEPENDENCIES
    if importlib.util.find_spec(import_name) is None
]
if still_missing:
    raise RuntimeError(
        "Packages were installed but still cannot be imported: "
        + ", ".join(still_missing)
        + ". Restart the kernel and rerun from this cell."
    )

if importlib.util.find_spec("nvalchemi") is None:
    print(
        "ALCHEMI Toolkit import is not visible yet. The Toolkit preflight below will "
        "give the authoritative error; install nvalchemi-toolkit packages only from "
        "the configured NVIDIA/private package index for this runtime."
    )

print("Notebook public dependencies are importable in this kernel.")


In [ ]:
# Notebook utility: print package versions for reproducibility.
import sys
from importlib.metadata import version

print(f"Python     : {sys.version.split()[0]}")
for pkg in ("ase", "numpy", "pandas", "matplotlib", "pymatgen", "pydantic",
            "ipywidgets", "tqdm", "ovito", "torch", "nvalchemi-toolkit"):
    try:
        print(f"{pkg:<17} : {version(pkg)}")
    except Exception as e:
        print(f"{pkg:<17} : NOT INSTALLED ({type(e).__name__})")


In [ ]:
# Import utilities for structure builders, result analysis, and visualization.
# The main Toolkit API appears explicitly in the H2O batching cells below.
from pathlib import Path
from dataclasses import asdict

import ase
import numpy as np
import pandas as pd
from ase.build import molecule as ase_molecule

# These utilities cover Toolkit wiring, structure primitives, plotting, widgets,
# and table formatting. The scientific panel, site/orientation choices, grid
# construction, and ranking rules are defined in notebook cells.
from helpers import (
    # Toolkit execution checks and selection
    RelaxationBatchResult, OptimizationResult,
    ToolkitRelaxationConfig,
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    get_toolkit_relaxation_engine,
    # Data conversion
    ase_to_atomic_data, atomic_data_to_ase,
    # Surface builders
    build_cu111_slab, build_cu100_slab, build_cu110_slab,
    build_tio2_110_slab, build_tio2_100_slab, build_tio2_101_slab,
    build_tin_001_slab, build_tin_110_slab, build_tin_210_slab,
    # Low-level site finders and configuration container
    Configuration,
    find_fcc_sites, find_elemental_surface_sites,
    find_oxide_sites, find_binary_surface_sites,
    # Surface-screen tables and path utilities
    SurfaceScreenSlabSpec, SurfaceScreenAdsorbateSpec,
    surface_screen_expected_counts, surface_screen_output_paths,
    surface_screen_result_artifact_paths, write_surface_screen_result_artifacts,
    safe_artifact_label, surface_screen_plan_table,
    require_surface_screen_artifact, surface_screen_result_json_path,
    audit_initial_configs, build_step_statistics, summarize_surface_screen_pairs,
    build_application_heatmap, build_difficult_cases,
    # Molecule builders
    build_co, build_h2o, build_nh3, build_methanol,
    # Slab masks
    make_active_mask,
    # Result analysis
    ADSORPTION_ENERGY_FORMULA,
    build_pair_results_table, summarize_pair_validation,
    # Visualization and plotting
    render_structure_ovito, render_trajectory_video_grid,
    display_widgets_row, display_widgets_grid,
    display_paged_widgets_grid, display_inline,
    make_notebook_progress,
    plot_h2o_batch_speedup, plot_adsorption_batch_sweep,
    plot_surface_screen_heatmap,
    # Batch-size sweep helpers
    display_adsorption_batch_sweep_results,
    build_adsorption_batch_sweep_pool,
    load_or_run_adsorption_batch_sweep,
    run_adsorption_batch_sweep,
    # Validation workflows
    make_oc20dense_validation_context, validation_context_table,
    run_or_load_oc20dense_validation,
    show_trajectory_validation_results, validation_trajectory_artifact_paths,
    show_nh3_ranking_results, show_nh3_geometry_widgets,
    show_nh3_all_geometry_grid, show_validation_model_tradeoff,
    # References used in validation/result discussion
    ADSORBML_REFERENCES, LITERATURE_OC157_MAD_GUIDE_EV, get_adsorbml_reference,
)


## Toolkit check

Run the next cells before any calculations. They confirm that the native ALCHEMI Toolkit API is available in the kernel, check CUDA runtime visibility when a GPU is requested, and construct the Toolkit relaxation engine used by the rest of the notebook.

In [ ]:
# Check that the Toolkit package is available before any expensive cells run.
status = check_toolkit_native_api()
print(status["message"])


In [ ]:
# Check the CUDA/PyTorch runtime before running Toolkit calculations.
import ctypes
import logging
import warnings
import torch

warnings.filterwarnings("ignore", message=".*torch.jit.script.*", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="To copy construct from a tensor.*", category=UserWarning)
warnings.filterwarnings(
    "ignore",
    message="The TorchScript type system doesn't support instance-level annotations.*",
    category=UserWarning,
)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)
try:
    # Some notebook/runtime combinations still enter TorchDynamo through
    # package-level optimized hooks even when TOOLKIT_COMPILE_MODEL is False.
    # If that compiler path is unavailable, fall back to eager execution instead
    # of failing the tutorial in a diagnostic/repro-generation step.
    torch._dynamo.config.suppress_errors = True
except Exception:
    pass
for name in ("_jit_set_texpr_fuser_enabled", "_jit_override_can_fuse_on_gpu"):
    if hasattr(torch._C, name):
        try:
            getattr(torch._C, name)(False)
        except TypeError:
            pass

if TOOLKIT_DEVICE != "cpu" and not (TOOLKIT_DEVICE == "auto" and not torch.cuda.is_available()):
    cuda_version = getattr(torch.version, "cuda", None)
    if cuda_version:
        major = cuda_version.split(".")[0]
        candidates = [f"libnvrtc-builtins.so.{major}.0", f"libnvrtc-builtins.so.{major}", "libnvrtc-builtins.so"]
        nvrtc_visible = False
        for library_name in candidates:
            try:
                ctypes.CDLL(library_name)
                nvrtc_visible = True
                break
            except OSError:
                pass
        if not nvrtc_visible:
            venv_cuda_lib = os.path.join(
                sys.prefix,
                "lib",
                f"python{sys.version_info.major}.{sys.version_info.minor}",
                "site-packages",
                "nvidia",
                f"cu{major}",
                "lib",
            )
            raise RuntimeError(
                "Toolkit CUDA execution needs NVRTC builtins visible to the dynamic linker. "
                "Restart Jupyter with the venv CUDA library path, for example:\n"
                f"LD_LIBRARY_PATH={venv_cuda_lib}:$LD_LIBRARY_PATH jupyter lab --no-browser --ip=127.0.0.1 --port=8888\n"
                f"Missing one of: {', '.join(candidates)}"
            )

print("Runtime preflight complete: expected PyTorch/MACE notebook warnings are filtered and CUDA NVRTC is visible.")

In [ ]:
print("Toolkit runtime preflight passed. H2O and adsorption sections build their model paths when needed.")
print(
    "Acceleration defaults: "
    f"compile_model={TOOLKIT_COMPILE_MODEL}, enable_cueq={TOOLKIT_ENABLE_CUEQ}"
)

---

## Batched H<sub>2</sub>O relaxation: Molecular "Hello World" in Toolkit

Before building surfaces, start with a controlled molecule-only example: many independent water molecules, relaxed together through the Toolkit.

The chemistry is intentionally simple so the batching pattern is visible. The model is loaded once, the structures are packed into one `Batch`, and the GPU processes many small systems together instead of repeating the same setup one structure at a time.

### Toolkit basics: molecule -> batch -> relaxation

Start with the chemistry object and build up one layer at a time. In the next cells, H<sub>2</sub>O is deliberately simple so the Toolkit pattern is the part to focus on:

1. make one ASE `Atoms` molecule;
2. make a short list of independent ASE structures;
3. convert each structure to Toolkit `AtomicData`;
4. pack the list into one GPU `Batch`;
5. load a MACE model with `MACEWrapper`;
6. relax the whole batch with `FIRE2`.

Only after this explicit run do we wrap the same steps into a small timing function and plot the speedup. That keeps the science and the Toolkit steps visible before the benchmarking machinery appears.


In [ ]:
# Step 1: construct the molecule and a small list of independent inputs.
from time import perf_counter

H2O_EXAMPLE_BATCH_SIZE = 4

# ASE is the molecule-building layer. It returns an Atoms object with element
# identities and Cartesian coordinates. For this molecule-only API example we
# keep it non-periodic; surface cells and gas-reference boxes appear later.
h2o_single = ase_molecule("H2O")
h2o_single.info["structure_id"] = "H2O_single"

print(f"One ASE object: H2O | atoms={len(h2o_single)} | pbc={h2o_single.pbc.tolist()}")
display_widgets_row(
    [("one H2O molecule", h2o_single)],
    width="260px",
    height="220px",
    show_cell=False,
)

# For a batch, we keep the structures independent. Each item could have a
# different molecule, orientation, site, or surface later in the tutorial.
h2o_atoms_list = []
for index in range(H2O_EXAMPLE_BATCH_SIZE):
    atoms = ase_molecule("H2O")
    atoms.info["structure_id"] = f"H2O_{index}"
    h2o_atoms_list.append(atoms)

print(f"Batch input: {len(h2o_atoms_list)} independent ASE structures")

In [ ]:
# Step 2: import the Toolkit building blocks used below.
import torch
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import ConvergenceHook
from nvalchemi.dynamics.hooks import NaNDetectorHook
from nvalchemi.dynamics.optimizers import FIRE2
from nvalchemi.models.mace import MACEWrapper



In [ ]:
# Step 3: convert ASE structures into Toolkit AtomicData, then pack a Batch.
if TOOLKIT_DEVICE == "auto":
    toolkit_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    toolkit_device = torch.device(TOOLKIT_DEVICE)

if toolkit_device.type == "cpu":
    print("Running on CPU; the batch interface is the same, but the speedup plot is only meaningful on GPU.")

toolkit_dtype = getattr(torch, TOOLKIT_DTYPE)
torch.set_float32_matmul_precision(TOOLKIT_MATMUL_PRECISION)

h2o_atomic_data = []
for atoms in h2o_atoms_list:
    # AtomicData is the Toolkit graph representation for one chemical system.
    # device="cpu" keeps construction lightweight; Batch.from_data_list moves
    # the packed batch to the selected device below.
    data = AtomicData.from_atoms(atoms, device="cpu", dtype=toolkit_dtype)

    # FIRE2 updates forces and energy during relaxation. Initial placeholders
    # make those fields explicit before the model evaluates them.
    data.forces = torch.zeros_like(data.positions)
    data.energy = torch.zeros(1, 1, dtype=toolkit_dtype)
    h2o_atomic_data.append(data)

# This is the batching step: a Python list of independent structures becomes
# one Toolkit Batch on the selected device.
h2o_batch = Batch.from_data_list(h2o_atomic_data, device=toolkit_device)

print(f"Toolkit device: {toolkit_device}")
print(f"AtomicData objects: {len(h2o_atomic_data)}")
print(f"Batch object: {type(h2o_batch).__name__}")


In [ ]:
# Step 4: load the model and relax the batch.
H2O_TOOLKIT_CHECKPOINT = "medium-mpa-0"  # default/general MACE checkpoint for the molecule-only example
H2O_DT = 0.01      # FIRE2 step size for this short molecular relaxation
H2O_FMAX = 0.05    # eV/A stopping criterion; max force below this value means converged
H2O_EXAMPLE_STEPS = 20

# MACEWrapper loads the MLIP through Toolkit. The same wrapper is reused for
# the timing sweep so first-call setup does not dominate every batch size.
h2o_model = MACEWrapper.from_checkpoint(
    H2O_TOOLKIT_CHECKPOINT,
    device=toolkit_device,
    dtype=toolkit_dtype,
    enable_cueq=TOOLKIT_ENABLE_CUEQ,
    compile_model=TOOLKIT_COMPILE_MODEL,
)
h2o_model.model_config.active_outputs = {"energy", "forces"}

# FIRE2 is the geometry optimizer. The convergence hook stops each structure
# once the largest atomic force is below H2O_FMAX.
h2o_optimizer = FIRE2(
    h2o_model,
    dt=H2O_DT,
    n_steps=H2O_EXAMPLE_STEPS,
    convergence_hook=ConvergenceHook.from_fmax(threshold=H2O_FMAX, source_status=0, target_status=1),
)
for hook in h2o_model.make_neighbor_hooks():
    h2o_optimizer.register_hook(hook)
h2o_optimizer.register_hook(NaNDetectorHook())

if str(toolkit_device).startswith("cuda"):
    torch.cuda.synchronize(toolkit_device)
h2o_relaxed_batch = h2o_optimizer.run(h2o_batch)
if str(toolkit_device).startswith("cuda"):
    torch.cuda.synchronize(toolkit_device)

h2o_rows = []
for idx, atoms in enumerate(h2o_atoms_list):
    relaxed = h2o_relaxed_batch.get_data(idx)
    energy = float(relaxed.energy.detach().cpu().reshape(-1)[0])
    forces = relaxed.forces.detach().cpu().numpy().reshape(-1, 3)
    h2o_rows.append(
        {
            "structure": atoms.info["structure_id"],
            "atoms": len(atoms),
            "energy_eV": energy,
            "fmax_eV_A": float(np.linalg.norm(forces, axis=1).max()),
        }
    )

print(f"Relaxed object: {type(h2o_relaxed_batch).__name__}")
h2o_example_df = pd.DataFrame(h2o_rows)
display(
    h2o_example_df.rename(
        columns={
            "structure": "structure",
            "atoms": "atoms",
            "energy_eV": "energy (eV)",
            "fmax_eV_A": "max force (eV/A)",
        }
    ).style.hide(axis="index").format(
        {"energy (eV)": "{:.6f}", "max force (eV/A)": "{:.4f}"}
    )
)


In [ ]:
# Step 5: turn the visible sequence into a small benchmark function.
# From here on, the code is machinery for the speedup figure; the Toolkit steps
# are the same ones shown above.
try:
    h2o_model
    toolkit_device
    toolkit_dtype
except NameError as exc:
    raise RuntimeError(
        "Run the H2O Toolkit cells in order first. Step 4 loads `h2o_model`, "
        "which is reused for the timing sweep so first-call setup does not "
        "dominate each batch size."
    ) from exc
H2O_BATCH_SIZES_BY_SCOPE = {
    "short": [1, 2, 4, 8, 16],
    "full": [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048],
}
H2O_BATCH_SIZES = H2O_BATCH_SIZES_BY_SCOPE[RUN_SCOPE]
H2O_BENCH_STEPS = {"short": 20, "full": 80}[RUN_SCOPE]
H2O_SPEEDUP_OUTPUT_DIR = (
    os.path.join(LIVE_OUTPUT_DIR, "tutorial")
    if USE_SAVED_TUTORIAL_RESULTS and not REFRESH_SAVED_RESULTS
    else TUTORIAL_OUTPUT_DIR
)
H2O_SPEEDUP_ROOT = os.path.join(H2O_SPEEDUP_OUTPUT_DIR, f"h2o_speedup_{RUN_SCOPE}")
H2O_SPEEDUP_CACHE = os.path.join(H2O_SPEEDUP_ROOT, "tables", "h2o_toolkit_batch_speedup.csv")
H2O_SPEEDUP_FIGURE = os.path.join(H2O_SPEEDUP_ROOT, "figures", "h2o_toolkit_batch_speedup.png")


def build_h2o_batch_atoms(batch_size: int) -> list[ase.Atoms]:
    """Build independent ASE H2O structures for one Toolkit timing batch."""
    atoms_list = []
    for index in range(batch_size):
        atoms = ase_molecule("H2O")
        atoms.info["structure_id"] = f"H2O_{index}"
        atoms_list.append(atoms)
    return atoms_list


def atoms_to_atomic_data(atoms: ase.Atoms):
    """Same ASE -> AtomicData conversion used above."""
    data = AtomicData.from_atoms(atoms, device="cpu", dtype=toolkit_dtype)
    data.forces = torch.zeros_like(data.positions)
    data.energy = torch.zeros(1, 1, dtype=toolkit_dtype)
    return data


def run_h2o_batch(batch_size: int) -> dict[str, float | int]:
    """Run one timed H2O Toolkit batch with the already-loaded model."""
    atoms_list = build_h2o_batch_atoms(batch_size)
    atomic_data = [atoms_to_atomic_data(atoms) for atoms in atoms_list]
    batch = Batch.from_data_list(atomic_data, device=toolkit_device)
    optimizer = FIRE2(
        h2o_model,
        dt=H2O_DT,
        n_steps=H2O_BENCH_STEPS,
        convergence_hook=ConvergenceHook.from_fmax(threshold=H2O_FMAX, source_status=0, target_status=1),
    )
    for hook in h2o_model.make_neighbor_hooks():
        optimizer.register_hook(hook)
    optimizer.register_hook(NaNDetectorHook())

    if str(toolkit_device).startswith("cuda"):
        torch.cuda.synchronize(toolkit_device)
    start = perf_counter()
    relaxed_batch = optimizer.run(batch)
    if str(toolkit_device).startswith("cuda"):
        torch.cuda.synchronize(toolkit_device)
    wall_time = perf_counter() - start

    energies = []
    for idx in range(batch_size):
        data = relaxed_batch.get_data(idx)
        energies.append(float(data.energy.detach().cpu().reshape(-1)[0]))

    return {
        "batch_size": batch_size,
        "wall_time_s": wall_time,
        "structures_per_s": batch_size / wall_time,
        "energy_mean_eV": float(np.mean(energies)),
        "energy_std_eV": float(np.std(energies)),
    }


print(
    f"H2O sweep: running {len(H2O_BATCH_SIZES)} batch sizes "
    f"up to {max(H2O_BATCH_SIZES)} structures."
)

In [ ]:
# Step 6: run the H2O batches, compute speedup, and display the figure.
# One unreported warm-up pass keeps the plot focused on batching rather
# than first-call CUDA/kernel setup.
warmup_result = run_h2o_batch(1)
h2o_speedup_rows = []
progress = make_notebook_progress(
    title="Toolkit batch sweep",
    total=len(H2O_BATCH_SIZES),
    unit="batch sizes",
    message="warmup complete; starting H2O batches",
    average_label="s/batch run",
    width_px=560,
)
for index, size in enumerate(H2O_BATCH_SIZES, start=1):
    progress.update(done=index - 1, message=f"running batch size {size}")
    h2o_speedup_rows.append(run_h2o_batch(size))
    progress.update(done=index, message=f"finished batch size {size}")

h2o_speedup_df = pd.DataFrame(h2o_speedup_rows)
os.makedirs(os.path.dirname(H2O_SPEEDUP_CACHE), exist_ok=True)
h2o_speedup_df.to_csv(H2O_SPEEDUP_CACHE, index=False)

single_time = float(h2o_speedup_df.loc[h2o_speedup_df["batch_size"] == 1, "wall_time_s"].iloc[0])
h2o_speedup_df["one_at_a_time_s"] = single_time * h2o_speedup_df["batch_size"]
h2o_speedup_df["speedup_vs_single"] = h2o_speedup_df["one_at_a_time_s"] / h2o_speedup_df["wall_time_s"]
E_H2O_molecule = float(h2o_speedup_df.loc[h2o_speedup_df["batch_size"] == 1, "energy_mean_eV"].iloc[0])
results = h2o_speedup_df.to_dict("records")

H2O_SPEEDUP_FIGURE = plot_h2o_batch_speedup(
    speedup_df=h2o_speedup_df,
    examples=[],
    output_path=H2O_SPEEDUP_FIGURE,
)
display_inline(H2O_SPEEDUP_FIGURE)

h2o_best = h2o_speedup_df.loc[h2o_speedup_df["structures_per_s"].idxmax()]
h2o_largest = h2o_speedup_df.sort_values("batch_size").iloc[-1]
h2o_summary = pd.DataFrame(
    [
        {
            "measured batch sizes": len(h2o_speedup_df),
            "fastest batch size": int(h2o_best["batch_size"]),
            "fastest throughput (structures/s)": float(h2o_best["structures_per_s"]),
            "largest-batch speedup": float(h2o_largest["speedup_vs_single"]),
            "E(H2O molecule) (eV)": E_H2O_molecule,
            "full CSV": tutorial_relpath(H2O_SPEEDUP_CACHE),
        }
    ]
)
print(f"Saved figure: {tutorial_relpath(H2O_SPEEDUP_FIGURE)}")
display(
    h2o_summary.style.hide(axis="index").format(
        {
            "fastest throughput (structures/s)": "{:.1f}",
            "largest-batch speedup": "{:.1f}x",
            "E(H2O molecule) (eV)": "{:.4f}",
        }
    )
)


### Adsorption batch-size sweep

The H<sub>2</sub>O figure isolates the batching idea. Research runs like adsorption batches are heavier: each item includes a slab, an adsorbate, periodic boundaries, frozen atoms, and neighbor lists. Before a longer search, it is useful to measure the practical throughput/memory trade-off for the actual chemistry.

Here we run a short H<sub>2</sub>O/TiO<sub>2</sub>(110) relaxation pool and compare open MACE choices:

- `MACE-MPA-0 medium`: the open default used for validation and the main adsorption screen later in the notebook.
- `MACE-MP-0 small`: a smaller open checkpoint that is useful for seeing the lower-memory, higher-throughput end of the trade-off.
- `MACE-MP-0 large`: a larger open checkpoint that gives the same workflow a different memory and throughput profile.

This tutorial uses MACE-MP/MACE-MPA checkpoints only. MACE-MH-1 with the OC20 surface head is the surface-specialized follow-up we would test next; its model card reports OC20/surface heads and surface-adsorption benchmarks. It is beyond the scope of this runnable NVIDIA tutorial because the upstream model is ASL-licensed, so no shipped outputs here are produced with MH-1. See the [MACE-MH-1 model card](https://huggingface.co/mace-foundations/mace-mh-1) and [MACE foundation-model repository](https://github.com/ACEsuit/mace-foundations).

A larger or more specialized model is not automatically too expensive for a batched workflow; likewise, larger model doesn't guarantee better accuracy. We measure memory and structures/s because the best batch size depends on the model, system size, and GPU. The sweep keeps increasing batch size until throughput flattens or memory becomes limiting; the recommended batch is the smallest measured batch near the best structures/s rate while staying under a realistic VRAM headroom rule.


In [ ]:
# First visual check: one H2O/TiO2(110) starting geometry before the surface batch-size sweep.
# The same structure-building path feeds the short performance sweep below;
# seeing the actual slab + adsorbate first makes the timing plot less abstract.
from helpers import build_adsorption_batch_sweep_pool

TIO2_PREVIEW_HOST = "TiO2(110)"
TIO2_PREVIEW_ADSORBATE = "H2O"

h2o_tio2_preview_pool = build_adsorption_batch_sweep_pool(
    host_name=TIO2_PREVIEW_HOST,
    adsorbate_name=TIO2_PREVIEW_ADSORBATE,
    rotations_deg=(0.0,),
    heights_A=(2.2,),
)
h2o_tio2_preview = h2o_tio2_preview_pool[0]
print(f"Preview structure: {h2o_tio2_preview.label} | atoms={len(h2o_tio2_preview.atoms)}")
display_widgets_row(
    [("H2O on TiO2(110) starting geometry", h2o_tio2_preview.atoms)],
    width="360px",
    height="300px",
    show_cell=True,
)


In [ ]:
# Adsorption batch-size sweep setup: short relaxations on a representative oxide-surface pool.
ADSORPTION_BATCH_SWEEP_HOST = "TiO2(110)"
ADSORPTION_BATCH_SWEEP_ADSORBATE = "H2O"
ADSORPTION_BATCH_SWEEP_TAG = "h2o_tio2_110_open_mace"
ADSORPTION_BATCH_SWEEP_BATCH_SIZES_BY_SCOPE = {
    "short": [4, 8, 12],
    "full": [4, 8, 12, 16, 24, 32, 48, 64, 96],
}
ADSORPTION_BATCH_SWEEP_BATCH_SIZES = ADSORPTION_BATCH_SWEEP_BATCH_SIZES_BY_SCOPE[RUN_SCOPE]
ADSORPTION_BATCH_SWEEP_STEPS = 40
ADSORPTION_BATCH_SWEEP_DT = 0.01  # short FIRE2 step size used only for this batch-size sweep
ADSORPTION_BATCH_SWEEP_MEMORY_FRACTION = 0.80
ADSORPTION_BATCH_SWEEP_MODELS = [
    {"label": "MACE-MPA-0 medium", "checkpoint": "medium-mpa-0", "head": None},
    {"label": "MACE-MP-0 small", "checkpoint": "small", "head": None},
    {"label": "MACE-MP-0 large", "checkpoint": "large", "head": None},
]
ADSORPTION_BATCH_SWEEP_ROOT = os.path.join(
    TUTORIAL_OUTPUT_DIR, f"adsorption_batch_sweep_{ADSORPTION_BATCH_SWEEP_TAG}_{RUN_SCOPE}"
)
ADSORPTION_BATCH_SWEEP_FULL_ROOT = os.path.join(
    TUTORIAL_OUTPUT_DIR, f"adsorption_batch_sweep_{ADSORPTION_BATCH_SWEEP_TAG}_full"
)
ADSORPTION_BATCH_SWEEP_CACHE = os.path.join(
    ADSORPTION_BATCH_SWEEP_ROOT, "tables", "adsorption_batch_sweep.csv"
)
ADSORPTION_BATCH_SWEEP_FULL_CACHE = os.path.join(
    ADSORPTION_BATCH_SWEEP_FULL_ROOT, "tables", "adsorption_batch_sweep.csv"
)
ADSORPTION_BATCH_SWEEP_FIGURE = os.path.join(
    ADSORPTION_BATCH_SWEEP_ROOT, "figures", "adsorption_batch_sweep.png"
)
ADSORPTION_BATCH_SWEEP_FULL_FIGURE = os.path.join(
    ADSORPTION_BATCH_SWEEP_FULL_ROOT, "figures", "adsorption_batch_sweep.png"
)

print(f"Sweep chemistry            : {ADSORPTION_BATCH_SWEEP_ADSORBATE} on {ADSORPTION_BATCH_SWEEP_HOST}")
print(f"Short relaxation cap       : {ADSORPTION_BATCH_SWEEP_STEPS} optimizer steps")
print(f"Batch sizes to test        : {ADSORPTION_BATCH_SWEEP_BATCH_SIZES}")
print("Models to test             : " + ", ".join(spec["label"] for spec in ADSORPTION_BATCH_SWEEP_MODELS))


In [ ]:
# Run or load the adsorption batch-size sweep, then show the performance figure.
from IPython.display import Markdown, display

(
    adsorption_batch_sweep_df,
    adsorption_batch_sweep_summary,
    adsorption_batch_sweep_cache_source,
) = load_or_run_adsorption_batch_sweep(
    use_precomputed=USE_SAVED_TUTORIAL_RESULTS,
    run_scope=RUN_SCOPE,
    batch_sizes=ADSORPTION_BATCH_SWEEP_BATCH_SIZES,
    cache_path=ADSORPTION_BATCH_SWEEP_CACHE,
    full_cache_path=ADSORPTION_BATCH_SWEEP_FULL_CACHE,
    tutorial_relpath=tutorial_relpath,
    host_name=ADSORPTION_BATCH_SWEEP_HOST,
    adsorbate_name=ADSORPTION_BATCH_SWEEP_ADSORBATE,
    model_specs=ADSORPTION_BATCH_SWEEP_MODELS,
    output_dir=TUTORIAL_OUTPUT_DIR,
    toolkit_device=("cuda" if TOOLKIT_DEVICE == "auto" else TOOLKIT_DEVICE),
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_dt=ADSORPTION_BATCH_SWEEP_DT,
    n_steps=ADSORPTION_BATCH_SWEEP_STEPS,
    memory_fraction=ADSORPTION_BATCH_SWEEP_MEMORY_FRACTION,
    compile_model=TOOLKIT_COMPILE_MODEL,
    enable_cueq=TOOLKIT_ENABLE_CUEQ,
    progress_factory=make_notebook_progress,
    display_fn=display,
    markdown_cls=Markdown,
)

batch_sweep_figure_path = ADSORPTION_BATCH_SWEEP_FIGURE
if USE_SAVED_TUTORIAL_RESULTS and not REFRESH_SAVED_RESULTS:
    batch_sweep_figure_path = os.path.join(
        LIVE_OUTPUT_DIR,
        "tutorial",
        "plots",
        f"adsorption_batch_sweep_{ADSORPTION_BATCH_SWEEP_TAG}_{RUN_SCOPE}.png",
    )

ADSORPTION_BATCH_SWEEP_FIGURE = plot_adsorption_batch_sweep(
    adsorption_batch_sweep_df,
    output_path=batch_sweep_figure_path,
    title="H$_2$O/TiO$_2$(110) adsorption batch-size sweep",
)
display_inline(ADSORPTION_BATCH_SWEEP_FIGURE)
print(f"Saved: {tutorial_relpath(ADSORPTION_BATCH_SWEEP_FIGURE)}")

display_adsorption_batch_sweep_results(
    adsorption_batch_sweep_df,
    adsorption_batch_sweep_summary,
    display_fn=display,
    markdown_cls=Markdown,
)
print(f"Detailed timing rows: {tutorial_relpath(adsorption_batch_sweep_cache_source)}")


### Model selection and validation: speed is not enough

The H<sub>2</sub>O/TiO<sub>2</sub>(110) example shows what batching changes computationally: a GPU can relax many molecule-on-surface structures at once instead of repeating the same small calculation one structure at a time. Before we use that throughput for a larger adsorption search, we need the model choice to be explicit.

For this tutorial, the practical model question is:

> **Can this open MACE model give useful answers for molecule-on-surface chemistry at the scale we want to run?**

That is a joint question about chemistry, speed, memory, licensing, and sometimes the researcher's preference. We keep the runnable path on open MACE models, then compare against released OC20Dense DFT data so the notebook has a reference point: the starting structures, relaxed DFT geometries, and adsorption-energy targets come from a published surface-chemistry benchmark, while the calculations below are rerun with the Toolkit/MACE path.

We use two compact checks:

- **Relaxation check:** replay three released OC20Dense adsorption trajectories from their published starting frames. For one H<sub>2</sub>O, one NH<sub>3</sub>, and one N<sub>2</sub> case, the notebook writes the MACE relaxation trajectory and compares its endpoint with the released DFT trajectory endpoint: final adsorption energy, adsorbate geometry, and RMSD.
- **Ranking check:** take 92 DFT-relaxed NH<sub>3</sub> configurations for the same surface system, evaluate them in one batched single-point calculation, and ask whether MACE preserves the same relative adsorption landscape. Energies are shifted to the DFT rank-1 geometry, so the comparison focuses on ranking rather than an arbitrary absolute zero.

H<sub>2</sub>O, NH<sub>3</sub>, and N<sub>2</sub> are used here because their neutral gas references are unambiguous. D3 corrections are available in Toolkit workflows, but they stay off in this validation because the OC20Dense reference targets used here follow the non-D3 OC20 convention.

`VALIDATION_RESULT_SOURCE` controls this section. The default is `"compute"`; set it to `"saved"` in the control panel when you want to inspect saved validation tables without rerunning the validation check.


In [ ]:
# Validation settings: choose the examples, ranks, and model used for this checkpoint.
from IPython.display import Markdown, display

VALIDATION_TOOLKIT_CHECKPOINT = "medium-mpa-0"
VALIDATION_TOOLKIT_HEAD = None

TRAJECTORY_VALIDATION_SELECTION = [
    {"system_id": "3_2070_48", "config_id": "rand64", "sid": 14966, "adsorbate": "H2O", "dft_rank": 1},
    {"system_id": "72_7104_115", "config_id": "rand27", "sid": 3469, "adsorbate": "NH3", "dft_rank": 1},
    {"system_id": "69_1615_2", "config_id": "rand2", "sid": 23355, "adsorbate": "N2", "dft_rank": 1},
]

NH3_RANKING_SYSTEM = "72_7104_115"
NH3_RANKING_PREVIEW_RANKS = [1, 4, 16, 92]
SHOW_ALL_NH3_RANKING = False

VALIDATION_SINGLE_POINT_BATCH_SIZE = 12
VALIDATION_TOOLKIT_N_STEPS = 200
VALIDATION_TOOLKIT_FMAX = 0.05  # eV/A force threshold for validation relaxations

validation_context = make_oc20dense_validation_context(
    tutorial_root=TUTORIAL_ROOT,
    accuracy_output_dir=ACCURACY_OUTPUT_DIR,
    result_source=VALIDATION_RESULT_SOURCE,
    checkpoint=VALIDATION_TOOLKIT_CHECKPOINT,
    head=VALIDATION_TOOLKIT_HEAD,
    trajectory_selection=TRAJECTORY_VALIDATION_SELECTION,
    nh3_system=NH3_RANKING_SYSTEM,
    preview_ranks=NH3_RANKING_PREVIEW_RANKS,
    show_all_nh3_ranking=SHOW_ALL_NH3_RANKING,
    single_point_batch_size=VALIDATION_SINGLE_POINT_BATCH_SIZE,
    n_steps=VALIDATION_TOOLKIT_N_STEPS,
    fmax=VALIDATION_TOOLKIT_FMAX,
)

VALIDATION_MODEL_LABEL = validation_context.model_label
OC20DENSE_SOURCE_ROOT = validation_context.oc20dense_data_root
TRAJECTORY_VALIDATION_ROOT = validation_context.trajectory_root
NH3_RANKING_ROOT = validation_context.nh3_ranking_root

validation_choices_df = validation_context_table(validation_context, relpath_fn=tutorial_relpath)
display(validation_choices_df.style.hide(axis="index"))


### Run the validation

With `VALIDATION_RESULT_SOURCE = "compute"`, the next cell reruns the compact OC20Dense checkpoint directly from the Toolkit/Python workflow. With `"saved"`, it shows the saved result folders that feed the display cells without recomputing them.


In [ ]:
# Compute the validation checkpoint, or show the saved sources selected above.
validation_runtime_df = run_or_load_oc20dense_validation(
    validation_context,
    progress_factory=make_notebook_progress,
    display_fn=None,
    relpath_fn=tutorial_relpath,
)

display(Markdown(
    "Elapsed times below are notebook-side work: Toolkit recomputation plus lookup "
    "and consistency checks against released DFT reference records. No DFT is run here."
))
display(validation_runtime_df)


In [ ]:
# Relaxation check: compare MACE-relaxed endpoints with released DFT trajectories.
from helpers.validation_workflows import (
    show_trajectory_validation_results,
    validation_trajectory_artifact_paths,
)

trajectory_validation = show_trajectory_validation_results(
    validation_context,
    display_fn=display,
    markdown_cls=Markdown,
)
trajectory_per_config = trajectory_validation["trajectory_per_config"]
trajectory_dft_reference = trajectory_validation["trajectory_dft_reference"]
trajectory_mace_results = trajectory_validation["trajectory_mace_results"]
trajectory_mace_refs = trajectory_validation["trajectory_mace_refs"]
trajectory_relaxation_errors = trajectory_validation["trajectory_relaxation_errors"]
max_trajectory_dft_target_delta_eV = trajectory_validation["max_trajectory_dft_target_delta_eV"]

# Resolve the DFT and Toolkit trajectory files used by the video render below.
trajectory_video_paths = validation_trajectory_artifact_paths(
    validation_context,
    trajectory_validation,
)
display(Markdown("#### DFT and Toolkit trajectory files for MP4 rendering"))
display(
    trajectory_video_paths.assign(
        dft_trajectory_path=lambda df: df["dft_trajectory_path"].map(tutorial_relpath),
        toolkit_trajectory_path=lambda df: df["toolkit_trajectory_path"].map(tutorial_relpath),
    ).style.hide(axis="index")
)


In [ ]:
# Optional: render the validation trajectories as browser-safe MP4s.
# This is off by default for the barebones tutorial path because it requires
# a working OVITO renderer and can take about a minute on the reference runtime.
import importlib
import os
import sys
from pathlib import Path

current_tutorial_root = Path.cwd().resolve()
if not (current_tutorial_root / "helpers" / "__init__.py").exists():
    raise RuntimeError("Start this kernel from the tutorial folder before rendering trajectories.")
TUTORIAL_ROOT = current_tutorial_root
sys.path[:] = [
    str(TUTORIAL_ROOT),
    *[
        entry
        for entry in sys.path
        if entry
        and Path(entry).resolve() != TUTORIAL_ROOT
        and not (Path(entry).resolve() / "alchemi-mace-adsorption-search.ipynb").exists()
    ],
]
for module_name in [name for name in list(sys.modules) if name == "helpers" or name.startswith("helpers.")]:
    del sys.modules[module_name]
importlib.invalidate_caches()

from helpers.visualization import (
    display_validation_trajectory_videos,
    render_validation_trajectory_videos,
)

RENDER_VALIDATION_TRAJECTORY_VIDEO = True
VALIDATION_TRAJECTORY_RENDERER = "visrtx"  # Set to "visrtx" on a configured RTX Linux runtime.
VALIDATION_TRAJECTORY_SAMPLES_PER_PIXEL = 12
VALIDATION_TRAJECTORY_TARGET_FRAMES = 240
VALIDATION_TRAJECTORY_FPS = 30
INSTALL_FFMPEG_IF_MISSING = False

if RENDER_VALIDATION_TRAJECTORY_VIDEO:
    VALIDATION_TRAJECTORY_VIDEO_DIR = os.path.join(
        ACCURACY_OUTPUT_DIR,
        "renders",
        "validation_trajectory_mp4_tiles_v12_centered",
    )
    trajectory_video_grid = render_validation_trajectory_videos(
        trajectory_video_paths,
        output_dir=VALIDATION_TRAJECTORY_VIDEO_DIR,
        renderer=VALIDATION_TRAJECTORY_RENDERER,
        samples_per_pixel=VALIDATION_TRAJECTORY_SAMPLES_PER_PIXEL,
        frames=VALIDATION_TRAJECTORY_TARGET_FRAMES,
        fps=VALIDATION_TRAJECTORY_FPS,
        width=960,
        height=720,
        install_ffmpeg_if_missing=INSTALL_FFMPEG_IF_MISSING,
        progress_factory=None,
    )
    display_validation_trajectory_videos(
        trajectory_video_grid,
        output_dir=VALIDATION_TRAJECTORY_VIDEO_DIR,
        relpath_fn=tutorial_relpath,
        width=420,
    )
else:
    display(Markdown(
        "Trajectory MP4 rendering is skipped in the barebones run. Set "
        "`RENDER_VALIDATION_TRAJECTORY_VIDEO = True` when the active kernel has "
        "OVITO rendering and FFmpeg available."
    ))


In [ ]:
# Fixed-geometry ranking: score all 92 NH3 DFT-relaxed final geometries in batches.
from helpers.validation_workflows import show_nh3_ranking_results

nh3_ranking_results = show_nh3_ranking_results(
    validation_context,
    display_fn=display,
    markdown_cls=Markdown,
)
ranking_dft_reference = nh3_ranking_results["ranking_dft_reference"]
ranking_dft_final_sp = nh3_ranking_results["ranking_dft_final_sp"]
nh3_ranking_all = nh3_ranking_results["nh3_ranking_all"]
rank1_summary = nh3_ranking_results["rank1_summary"]
max_ranking_dft_target_delta_eV = nh3_ranking_results["max_ranking_dft_target_delta_eV"]
max_ranking_start_adsorbate_rmsd_A = nh3_ranking_results["max_ranking_start_adsorbate_rmsd_A"]


In [ ]:
# Educational checkpoint: stack 92 NH3 DFT-final geometries into one Toolkit Batch and score them.
import importlib
from pathlib import Path
from time import perf_counter

import torch
from ase.io import read as ase_read
from nvalchemi.data import AtomicData, Batch
from nvalchemi.models.mace import MACEWrapper
from nvalchemi.neighbors import compute_neighbors

import helpers.validation_workflows as validation_workflows

validation_workflows = importlib.reload(validation_workflows)
show_nh3_all_geometry_render_grid = validation_workflows.show_nh3_all_geometry_render_grid

# This is the same real 92-configuration validation set used in the ranking table.
nh3_92_table = ranking_dft_final_sp.sort_values("dft_rank").reset_index(drop=True)
nh3_92_atoms = []
for row in nh3_92_table.itertuples(index=False):
    atoms = ase_read(Path(row.dft_final_structure_path))
    atoms.info["structure_id"] = f"NH3_dft_final_rank_{int(row.dft_rank):03d}_{row.config_id}_sid{int(row.sid)}"
    nh3_92_atoms.append(atoms)

if len(nh3_92_atoms) != 92:
    raise RuntimeError(f"Expected 92 NH3 DFT-final geometries, found {len(nh3_92_atoms)}.")

# Make the core Toolkit API visible before the visual scan. The important
# teaching point is the transition from a Python list of ASE objects to one
# Toolkit Batch, followed by one neighbor-list build and one MACE model call.
display(Markdown("#### NH<sub>3</sub> 92-configuration single-batch Toolkit API"))
display(Markdown(
    "This cell loads the 92 released DFT-final NH<sub>3</sub> geometries, "
    "converts each structure to `AtomicData`, stacks all of them with "
    "`Batch.from_data_list(...)`, builds the batched neighbor list, and calls "
    "MACE once. The render below is only for inspection; the API block here is "
    "the computational lesson."
))
display(Markdown(
    "```python\n"
    "# ASE structures: one real DFT-final geometry per NH3 configuration\n"
    "nh3_92_atoms = [ase_read(Path(row.dft_final_structure_path)) for row in nh3_92_table.itertuples(index=False)]\n\n"
    "# Toolkit graph objects, still one object per structure\n"
    "nh3_atomic_data = [AtomicData.from_atoms(atoms, device='cpu', dtype=toolkit_dtype) for atoms in nh3_92_atoms]\n\n"
    "# One batched graph object on the selected runtime device\n"
    "nh3_batch = Batch.from_data_list(nh3_atomic_data, device=toolkit_device)\n\n"
    "# One batched neighbor-list build and one MACE call\n"
    "compute_neighbors(nh3_batch, config=nh3_model.model_config.neighbor_config)\n"
    "nh3_outputs = nh3_model(nh3_batch)\n"
    "nh3_energies = output_tensor(nh3_outputs, 'energy').reshape(-1)  # 92 energies\n"
    "```"
))

if TOOLKIT_DEVICE == "auto":
    toolkit_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    toolkit_device = torch.device(TOOLKIT_DEVICE)
toolkit_dtype = getattr(torch, TOOLKIT_DTYPE)
torch.set_float32_matmul_precision(TOOLKIT_MATMUL_PRECISION)

nh3_atomic_data = []
for atoms in nh3_92_atoms:
    data = AtomicData.from_atoms(atoms, device="cpu", dtype=toolkit_dtype)
    # MACE forces are returned as energy derivatives; keep gradients enabled.
    data.forces = torch.zeros_like(data.positions)
    data.energy = torch.zeros(1, 1, dtype=toolkit_dtype)
    nh3_atomic_data.append(data)

nh3_batch = Batch.from_data_list(nh3_atomic_data, device=toolkit_device)
nh3_model = MACEWrapper.from_checkpoint(
    validation_context.checkpoint,
    device=toolkit_device,
    dtype=toolkit_dtype,
    enable_cueq=TOOLKIT_ENABLE_CUEQ,
    compile_model=TOOLKIT_COMPILE_MODEL,
)
nh3_model.model_config.active_outputs = {"energy", "forces"}

def output_tensor(outputs, name: str):
    if isinstance(outputs, dict):
        value = outputs.get(name)
    else:
        value = getattr(outputs, name, None)
    if value is None:
        raise RuntimeError(f"MACE output is missing `{name}`.")
    return value

try:
    if str(toolkit_device).startswith("cuda"):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(toolkit_device)
        torch.cuda.synchronize(toolkit_device)
    start = perf_counter()
    compute_neighbors(nh3_batch, config=nh3_model.model_config.neighbor_config)
    nh3_outputs = nh3_model(nh3_batch)
    if str(toolkit_device).startswith("cuda"):
        torch.cuda.synchronize(toolkit_device)
    nh3_single_batch_wall_s = perf_counter() - start

    nh3_energies = output_tensor(nh3_outputs, "energy").detach().cpu().numpy().reshape(-1)
    nh3_forces = output_tensor(nh3_outputs, "forces").detach().cpu().numpy().reshape(-1, 3)
    if len(nh3_energies) != len(nh3_92_atoms):
        raise RuntimeError(f"Expected 92 MACE energies, got {len(nh3_energies)}.")

    batch_ptr = getattr(nh3_batch, "batch_ptr", None)
    if batch_ptr is None:
        offsets = np.cumsum([0, *[len(atoms) for atoms in nh3_92_atoms]])
    else:
        offsets = batch_ptr.detach().cpu().numpy().astype(int)

    nh3_single_batch_rows = []
    for index, row in enumerate(nh3_92_table.itertuples(index=False)):
        force_block = nh3_forces[offsets[index]: offsets[index + 1]]
        nh3_single_batch_rows.append({
            "reference rank": int(row.dft_rank),
            "config_id": row.config_id,
            "sid": int(row.sid),
            "atoms": len(nh3_92_atoms[index]),
            "MACE single-point total energy (eV)": float(nh3_energies[index]),
            "max force (eV/A)": float(np.linalg.norm(force_block, axis=1).max()),
        })
    nh3_single_batch_results = pd.DataFrame(nh3_single_batch_rows)
    nh3_single_batch_results["MACE single-point rank"] = (
        nh3_single_batch_results["MACE single-point total energy (eV)"]
        .rank(method="first", ascending=True)
        .astype(int)
    )
    nh3_single_batch_summary = pd.DataFrame([
        {
            "ASE structures loaded": len(nh3_92_atoms),
            "AtomicData objects stacked": len(nh3_atomic_data),
            "Toolkit object": type(nh3_batch).__name__,
            "MACE model calls": 1,
            "energies returned": len(nh3_energies),
            "device": str(toolkit_device),
            "total atoms": int(sum(len(atoms) for atoms in nh3_92_atoms)),
            "wall time (s)": nh3_single_batch_wall_s,
            "peak CUDA memory (GB)": (
                torch.cuda.max_memory_allocated(toolkit_device) / 1e9
                if str(toolkit_device).startswith("cuda")
                else np.nan
            ),
        }
    ])
    display(nh3_single_batch_summary.style.hide(axis="index").format({
        "wall time (s)": "{:.2f}",
        "peak CUDA memory (GB)": "{:.2f}",
    }))
    display(Markdown(
        "The table confirms the batching semantics: 92 ASE structures become 92 "
        "`AtomicData` objects, then one Toolkit `Batch`; one MACE call returns 92 "
        "single-point energies plus forces for all atoms. This is not DFT and not "
        "a geometry relaxation."
    ))
    display(
        nh3_single_batch_results.sort_values("MACE single-point rank")
        .head(8)
        .style.hide(axis="index")
        .format({
            "MACE single-point total energy (eV)": "{:.6f}",
            "max force (eV/A)": "{:.4f}",
        })
    )
    del nh3_outputs
    if str(toolkit_device).startswith("cuda"):
        torch.cuda.empty_cache()
except RuntimeError as exc:
    if "out of memory" not in str(exc).lower():
        raise
    if str(toolkit_device).startswith("cuda"):
        torch.cuda.empty_cache()
    nh3_single_batch_results = pd.DataFrame()
    display(Markdown(
        "The explicit 92-structure `Batch.from_data_list(...)` call was made, but "
        "this runtime did not have enough GPU memory for one-shot MACE inference. "
        "The production validation path above therefore uses the same API in smaller "
        f"batches of `{validation_context.single_point_batch_size}` structures."
    ))

# Visual inspection: render all 92 NH3 DFT-final geometries as a static OVITO scan.
RENDER_NH3_92_STATIC_GRID = True
NH3_92_RENDER_BACKGROUND = (0.0, 0.0, 0.0)

if RENDER_NH3_92_STATIC_GRID:
    nh3_all_geometry_render_paths = show_nh3_all_geometry_render_grid(
        validation_context,
        nh3_ranking_results,
        display_fn=display,
        markdown_cls=Markdown,
        renderer="visrtx",
        samples_per_pixel=24,
        tile_size=(640, 480),
        columns=4,
        show_cell=True,
        focus="full_slab",
        force=False,
        display_width=2600,
        annotation_scale=2.0,
        render_background=NH3_92_RENDER_BACKGROUND,
        overview_background=(224, 228, 233),
    )
else:
    nh3_all_geometry_render_paths = {}
    display(Markdown(
        "NH<sub>3</sub> 92-configuration static rendering is skipped in the "
        "barebones run. Set `RENDER_NH3_92_STATIC_GRID = True` on a configured "
        "OVITO/VisRTX runtime to regenerate the inspection PNG."
    ))


---

## Build the adsorption panel

The model path has already been introduced and checked, so the adsorption screen can now use the open `medium-mpa-0` MACE-MPA checkpoint and focus on the chemistry: which surfaces, which adsorbates, and which starting geometries should be relaxed in batches.

The structure-building stack is intentionally standard:

- **ASE** builds the Cu fcc slabs and the small molecule objects used as adsorbates.
- **pymatgen** builds rutile TiO<sub>2</sub> and rocksalt TiN bulk structures, cuts Miller-index slabs with `SlabGenerator`, and then hands them back as ASE `Atoms`.
- **ALCHEMI Toolkit** receives those ASE structures as `AtomicData`, packs them into `Batch` objects, and relaxes the independent starts on the GPU.

The panel is a compact teaching-scale version of an early R&D screen:

- one **metal family**: Cu low-index terraces, where the Miller index changes the coordination of surface atoms;
- one **oxide family**: rutile TiO<sub>2</sub>, where different cuts expose different Ti/O motifs;
- one **nitride/ceramic family**: rocksalt TiN, including a stepped high-index cut.

For each surface, we test CO, H<sub>2</sub>O, NH<sub>3</sub>, and CH<sub>3</sub>OH. Each adsorbate/surface pair uses three site classes and two molecular orientations, giving six starting geometries per pair.


In [ ]:
# Metal family: ASE fcc surface builders create three Cu facets with distinct coordination.
CU_SLAB_SPECS = [
    SurfaceScreenSlabSpec(
        name="Cu(111)", material_class="fcc metal", facet="close-packed terrace",
        miller_index=(1, 1, 1), builder_name="build_cu111_slab",
        builder=lambda: build_cu111_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(3, 3, 1)),
        default_supercell=(3, 3, 1),
        note="close-packed low-index Cu surface",
    ),
    SurfaceScreenSlabSpec(
        name="Cu(100)", material_class="fcc metal", facet="square terrace",
        miller_index=(1, 0, 0), builder_name="build_cu100_slab",
        builder=lambda: build_cu100_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(3, 3, 1)),
        default_supercell=(3, 3, 1),
        note="square low-index Cu surface",
    ),
    SurfaceScreenSlabSpec(
        name="Cu(110)", material_class="fcc metal", facet="open row surface",
        miller_index=(1, 1, 0), builder_name="build_cu110_slab",
        builder=lambda: build_cu110_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(3, 3, 1)),
        default_supercell=(3, 3, 1),
        note="more open low-index Cu surface",
    ),
]
CU_HOSTS = {spec.name: spec.builder() for spec in CU_SLAB_SPECS}

display(surface_screen_plan_table(CU_SLAB_SPECS).style.hide(axis="index"))
display_widgets_row([(name, atoms) for name, atoms in CU_HOSTS.items()], width="230px", height="210px", show_cell=True)


### Add rutile TiO<sub>2</sub> facets

The oxide block keeps the composition fixed and changes the surface cut. The builders call pymatgen's `SlabGenerator` on a rutile TiO<sub>2</sub> bulk structure, then convert the selected slab to ASE `Atoms` for the rest of the workflow.


In [ ]:
# Oxide family: pymatgen SlabGenerator cuts three rutile TiO2 facets.
TIO2_SLAB_SPECS = [
    SurfaceScreenSlabSpec(
        name="TiO2(110)", material_class="oxide", facet="rutile bridge-row surface",
        miller_index=(1, 1, 0), builder_name="build_tio2_110_slab",
        builder=lambda: build_tio2_110_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(2, 2, 1)),
        default_supercell=(2, 2, 1),
        note="canonical rutile surface with rows of under-coordinated Ti/O sites",
    ),
    SurfaceScreenSlabSpec(
        name="TiO2(100)", material_class="oxide", facet="rutile side surface",
        miller_index=(1, 0, 0), builder_name="build_tio2_100_slab",
        builder=lambda: build_tio2_100_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(2, 2, 1)),
        default_supercell=(2, 2, 1),
        note="second low-index rutile cut",
    ),
    SurfaceScreenSlabSpec(
        name="TiO2(101)", material_class="oxide", facet="rutile oblique surface",
        miller_index=(1, 0, 1), builder_name="build_tio2_101_slab",
        builder=lambda: build_tio2_101_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(2, 2, 1)),
        default_supercell=(2, 2, 1),
        note="oblique rutile cut; useful contrast to (110) and (100)",
    ),
]
TIO2_HOSTS = {spec.name: spec.builder() for spec in TIO2_SLAB_SPECS}

display(surface_screen_plan_table(TIO2_SLAB_SPECS).style.hide(axis="index"))
display_widgets_row([(name, atoms) for name, atoms in TIO2_HOSTS.items()], width="230px", height="210px", show_cell=True)


In [ ]:
# Nitride/ceramic family: pymatgen SlabGenerator cuts two nonpolar TiN terraces and one stepped cut.
TIN_SLAB_SPECS = [
    SurfaceScreenSlabSpec(
        name="TiN(001)", material_class="nitride ceramic", facet="rocksalt nonpolar terrace",
        miller_index=(0, 0, 1), builder_name="build_tin_001_slab",
        builder=lambda: build_tin_001_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(2, 2, 1)),
        default_supercell=(2, 2, 1),
        note="nonpolar rocksalt TiN terrace",
    ),
    SurfaceScreenSlabSpec(
        name="TiN(110)", material_class="nitride ceramic", facet="rocksalt rectangular terrace",
        miller_index=(1, 1, 0), builder_name="build_tin_110_slab",
        builder=lambda: build_tin_110_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(2, 2, 1)),
        default_supercell=(2, 2, 1),
        note="second nonpolar TiN low-index cut",
    ),
    SurfaceScreenSlabSpec(
        name="TiN(210)", material_class="nitride ceramic", facet="rocksalt stepped surface",
        miller_index=(2, 1, 0), builder_name="build_tin_210_slab",
        builder=lambda: build_tin_210_slab(min_slab_size=8.0, min_vacuum_size=15.0, supercell=(2, 2, 1)),
        default_supercell=(2, 2, 1),
        note="stepped/high-index TiN cut; TiN(111) is avoided because it is polar",
    ),
]
TIN_HOSTS = {spec.name: spec.builder() for spec in TIN_SLAB_SPECS}

display(surface_screen_plan_table(TIN_SLAB_SPECS).style.hide(axis="index"))
display_widgets_row([(name, atoms) for name, atoms in TIN_HOSTS.items()], width="230px", height="210px", show_cell=True)


In [ ]:
# Assemble the panel after each family has been defined and inspected.
SURFACE_SCREEN_SLABS = CU_SLAB_SPECS + TIO2_SLAB_SPECS + TIN_SLAB_SPECS
HOSTS = {**CU_HOSTS, **TIO2_HOSTS, **TIN_HOSTS}
HOST_NAMES = [spec.name for spec in SURFACE_SCREEN_SLABS]
HOST_COMPOSITIONS = {spec.name: spec.name.split("(")[0] for spec in SURFACE_SCREEN_SLABS}
HOST_MILLER_INDICES = {spec.name: spec.miller_index for spec in SURFACE_SCREEN_SLABS}
HOST_RELAXED = {name: atoms.copy() for name, atoms in HOSTS.items()}
HOST_RELAXED_ENERGIES = {}

host_plan_df = pd.DataFrame([
    {
        "surface": spec.name,
        "class": spec.material_class,
        "Miller index": str(spec.miller_index),
        "facet model": spec.facet,
        "atoms": len(HOSTS[spec.name]),
        "note": spec.note,
    }
    for spec in SURFACE_SCREEN_SLABS
])
display(host_plan_df.style.hide(axis="index"))

# Sanity check: the planned panel size is explicit before any Toolkit call.
SURFACE_SCREEN_COUNTS = surface_screen_expected_counts(
    n_slabs=len(SURFACE_SCREEN_SLABS),
    n_adsorbates=4,
    starts_per_pair=6,
)
for key, value in SURFACE_SCREEN_COUNTS.items():
    print(f"{key:<28}: {value}")


## Choose molecular probes

The adsorbates are small, closed-shell probes that appear in many surface-chemistry workflows. They give a compact way to exercise different bonding motifs: carbonyl binding, water adsorption, ammonia donation, and alcohol adsorption.


In [ ]:
# Visible adsorbate definition: two starting orientations per molecule.
# Molecule coordinates are built through helper functions in helpers.config_search,
# not token-written in this notebook. Those helpers return ASE Atoms: CO is a
# direct ase.Atoms object, while H2O, NH3, and CH3OH use ase.build.molecule and
# are reoriented so the named atom/group is the surface-facing end.
# "Down" is a placement convention: the named atom or group is the surface-facing
# end after the molecule is placed above a slab. It is not the notebook widget's
# screen-down direction.
from IPython.display import HTML
from helpers.visualization import create_interactive_view, subscript_formula_html
import ipywidgets as widgets

SURFACE_SCREEN_ADSORBATES = [
    SurfaceScreenAdsorbateSpec("CO", ("C-down", "O-down"), "CO binding and catalysis probe", "linear molecule with two contact atoms"),
    SurfaceScreenAdsorbateSpec("H2O", ("O-down", "H-down"), "water adsorption and hydrophilicity", "closed-shell water adsorption example"),
    SurfaceScreenAdsorbateSpec("NH3", ("N-down", "H-down"), "ammonia binding and nitrogen chemistry", "lone-pair donor with H-down decoys"),
    SurfaceScreenAdsorbateSpec("CH3OH", ("O-down", "methyl-down"), "alcohol adsorption and separations proxy", "larger polar molecule with orientation sensitivity"),
]
ADSORBATES = [spec.name for spec in SURFACE_SCREEN_ADSORBATES]
ADSORBATE_ORIENTATIONS_FOR_SCREEN = {spec.name: list(spec.orientations) for spec in SURFACE_SCREEN_ADSORBATES}
ADSORBATE_HINTS = {spec.name: spec.application_hint for spec in SURFACE_SCREEN_ADSORBATES}
ADSORBATE_BUILDERS = {"CO": build_co, "H2O": build_h2o, "NH3": build_nh3, "CH3OH": build_methanol}

ADSORBATE_DISPLAY_HTML = {
    "CO": "CO",
    "H2O": "H<sub>2</sub>O",
    "NH3": "NH<sub>3</sub>",
    "CH3OH": "CH<sub>3</sub>OH",
}
ORIENTATION_SURFACE_FACING_END = {
    "C-down": "C atom faces the surface",
    "O-down": "O atom faces the surface",
    "H-down": "H side faces the surface",
    "N-down": "N atom faces the surface",
    "methyl-down": "methyl group faces the surface",
}
orientation_convention_df = pd.DataFrame([
    {
        "adsorbate": ADSORBATE_DISPLAY_HTML[spec.name],
        "orientation label": orient,
        "meaning of down": ORIENTATION_SURFACE_FACING_END[orient],
    }
    for spec in SURFACE_SCREEN_ADSORBATES
    for orient in spec.orientations
])
display(HTML(
    "<div style='font-size: 200%; line-height: 1.2; margin: 0 0 10px 0;'>"
    "<strong>Orientation convention:</strong> in labels such as <code>N-down</code>, "
    "<strong>down means the surface-facing end after placement along the slab normal</strong>. "
    "It does not mean lower on the notebook screen; the molecule preview widget can be rotated freely."
    "</div>"
))
display(
    orientation_convention_df
    .style
    .format(escape=None)
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "200%"), ("line-height", "1.15"), ("padding", "8px 12px")]},
        {"selector": "td", "props": [("font-size", "200%"), ("line-height", "1.15"), ("padding", "8px 12px")]},
    ])
)


def standalone_adsorbate_preview(atoms):
    """Return a molecule-only display copy with no artificial periodic box."""
    preview = atoms.copy()
    preview.set_cell([0.0, 0.0, 0.0])
    preview.set_pbc(False)
    preview.translate(-preview.get_center_of_mass())
    return preview


def display_large_adsorbate_preview_grid(items, *, width="500px", height="220px", columns=4):
    cards = []
    for label, atoms in items:
        widget = create_interactive_view(atoms, width=width, height=height, show_cell=False)
        if widget is None:
            cards.append(widgets.HTML(f"<div style='font-size: 200%; font-weight: 700;'>{subscript_formula_html(label)}</div>"))
            continue
        cards.append(widgets.VBox(
            [
                widgets.HTML(
                    "<div style='font-size: 200%; font-weight: 700; line-height: 1.15; "
                    f"margin: 0 0 8px 0;'>{subscript_formula_html(label)}</div>"
                ),
                widget,
            ],
            layout=widgets.Layout(width=width, min_width=width, gap="4px"),
        ))
    display(widgets.GridBox(
        cards,
        layout=widgets.Layout(
            grid_template_columns=f"repeat({int(columns)}, {width})",
            grid_gap="18px 20px",
            align_items="flex-start",
            width=f"calc({int(columns)} * {width} + {(int(columns) - 1) * 20}px)",
        ),
    ))


sample_adsorbates = []
for spec in SURFACE_SCREEN_ADSORBATES:
    for orient in spec.orientations:
        ads = standalone_adsorbate_preview(ADSORBATE_BUILDERS[spec.name](orient))
        sample_adsorbates.append((
            f"{spec.name}: {orient} ({ORIENTATION_SURFACE_FACING_END[orient]})",
            ads,
        ))

display_large_adsorbate_preview_grid(sample_adsorbates)


## Define the starting-geometry search

For each adsorbate/surface pair, the screen uses three site classes and two orientations. That is a small AdsorbML-style local search space: enough to make the starting-point assumption measurable, but still small enough to discuss in a live tutorial.

The starting height is 2.8 Å. That is intentionally conservative for corrugated oxide and nitride surfaces: H-down and methyl-down decoy orientations should start above the surface, not buried inside it.


In [ ]:
# Site classes are chosen explicitly for each surface family.
# The site-finder mapping stays visible so the chemistry is not hidden in a helper.
# The coordinates themselves are still generated through helper functions from
# the active ASE slab geometry: Cu slabs come from ASE surface builders, while
# TiO2 and TiN slabs come from pymatgen SlabGenerator and are converted to ASE
# before these site finders run.
SURFACE_SITES_BY_HOST = {
    "Cu(111)": ["top", "bridge", "fcc"],
    "Cu(100)": ["top", "bridge", "hollow"],
    "Cu(110)": ["top", "bridge", "hollow"],
    "TiO2(110)": ["ti-top", "o-top", "bridge"],
    "TiO2(100)": ["ti-top", "o-top", "bridge"],
    "TiO2(101)": ["ti-top", "o-top", "bridge"],
    "TiN(001)": ["ti-top", "n-top", "bridge"],
    "TiN(110)": ["ti-top", "n-top", "bridge"],
    "TiN(210)": ["ti-top", "n-top", "bridge"],
}
SURFACE_SITE_FINDERS = {
    "Cu(111)": find_fcc_sites,
    "Cu(100)": find_elemental_surface_sites,
    "Cu(110)": find_elemental_surface_sites,
    "TiO2(110)": lambda slab: find_oxide_sites(slab, cation_symbol="Ti", cation_site_name="ti-top"),
    "TiO2(100)": lambda slab: find_oxide_sites(slab, cation_symbol="Ti", cation_site_name="ti-top"),
    "TiO2(101)": lambda slab: find_oxide_sites(slab, cation_symbol="Ti", cation_site_name="ti-top"),
    "TiN(001)": lambda slab: find_binary_surface_sites(
        slab, cation_symbol="Ti", cation_site_name="ti-top", anion_symbol="N", anion_site_name="n-top"
    ),
    "TiN(110)": lambda slab: find_binary_surface_sites(
        slab, cation_symbol="Ti", cation_site_name="ti-top", anion_symbol="N", anion_site_name="n-top"
    ),
    "TiN(210)": lambda slab: find_binary_surface_sites(
        slab, cation_symbol="Ti", cation_site_name="ti-top", anion_symbol="N", anion_site_name="n-top"
    ),
}

# Site coordinates are computed after clean-slab relaxation/loading, so every
# starting geometry uses positions from the same slab geometry used for Eads.
SURFACE_SITE_CANDIDATES_BY_HOST = {}

ADSORPTION_START_HEIGHT_A = 2.8
ADSORPTION_ROTATIONS_DEG = (0.0,)
FROZEN_SLAB_FRACTION = 0.5
RELIABILITY_MAX_FORCE_EV_A = 0.2
DESORPTION_HEIGHT_A = 5.0
NOMINATED_SINGLE_START_BY_HOST = {
    host: sites[0]
    for host, sites in SURFACE_SITES_BY_HOST.items()
}

site_plan_df = pd.DataFrame([
    {
        "surface": host,
        "site classes": ", ".join(sites),
        "nominated single-start site": NOMINATED_SINGLE_START_BY_HOST[host],
    }
    for host, sites in SURFACE_SITES_BY_HOST.items()
])
orientation_plan_df = pd.DataFrame([
    {"adsorbate": spec.name, "orientations": ", ".join(spec.orientations), "why included": spec.note}
    for spec in SURFACE_SCREEN_ADSORBATES
])

display(site_plan_df.style.hide(axis="index"))
display(orientation_plan_df.style.hide(axis="index"))
print(f"Starting height: {ADSORPTION_START_HEIGHT_A} A; rotations: {ADSORPTION_ROTATIONS_DEG}; frozen slab fraction: {FROZEN_SLAB_FRACTION}")
print(f"Reliability filter: converged, adsorbed, max force <= {RELIABILITY_MAX_FORCE_EV_A} eV/A; desorbed if closest adsorbate atom is > {DESORPTION_HEIGHT_A} A above the slab.")


In [ ]:
# Starting-geometry construction: one relaxed slab, one helper-built molecule,
# and one helper-selected surface site. This cell combines the pieces; it does
# not define new molecular or slab coordinates by hand.
def surface_normal_from_cell(slab: ase.Atoms) -> np.ndarray:
    normal = np.cross(slab.cell[0], slab.cell[1])
    return normal / np.linalg.norm(normal)


def place_adsorbate_for_start(
    slab: ase.Atoms,
    adsorbate_atoms: ase.Atoms,
    site_position: np.ndarray,
    *,
    height_A: float,
    rotation_deg: float,
) -> ase.Atoms:
    ads = adsorbate_atoms.copy()
    if abs(rotation_deg) > 1e-6:
        ads.rotate(rotation_deg, "z", center=(0.0, 0.0, 0.0))
    ads.translate(site_position + height_A * surface_normal_from_cell(slab))
    return slab.copy() + ads


def build_adsorption_starting_grid(
    *,
    host: str,
    adsorbate: str,
    site_names: list[str] | None = None,
    orientation_names: list[str] | None = None,
) -> list[Configuration]:
    slab = HOST_RELAXED[host]
    site_names = list(site_names or SURFACE_SITES_BY_HOST[host])
    orientation_names = list(orientation_names or ADSORBATE_ORIENTATIONS_FOR_SCREEN[adsorbate])
    base_mask = make_active_mask(slab, bottom_fraction=FROZEN_SLAB_FRACTION)

    configs = []
    for site_name in site_names:
        for site_position in SURFACE_SITE_CANDIDATES_BY_HOST[host][site_name]:
            for orientation in orientation_names:
                adsorbate_atoms = ADSORBATE_BUILDERS[adsorbate](orientation)
                for rotation_deg in ADSORPTION_ROTATIONS_DEG:
                    atoms = place_adsorbate_for_start(
                        slab,
                        adsorbate_atoms,
                        site_position,
                        height_A=ADSORPTION_START_HEIGHT_A,
                        rotation_deg=rotation_deg,
                    )
                    configs.append(Configuration(
                        label=(
                            f"{adsorbate}_{host}_{site_name}_{orientation}_"
                            f"rot{int(rotation_deg)}_h{ADSORPTION_START_HEIGHT_A:.1f}"
                        ),
                        host=host,
                        adsorbate=adsorbate,
                        site=site_name,
                        orientation=orientation,
                        rot_deg=float(rotation_deg),
                        height=float(ADSORPTION_START_HEIGHT_A),
                        atoms=atoms,
                        active_mask=base_mask + [True] * len(adsorbate_atoms),
                    ))
    return configs


## Clean-slab and gas references

Adsorption energies are only meaningful after the reference terms are fixed. The notebook relaxes each clean slab with the same frozen-layer convention, and each isolated molecule in a vacuum box. The adsorption-energy convention used later is:

`E_ads = E(slab + adsorbate) - E(clean slab) - E(gas molecule)`

Negative values are exothermic binding energies in eV per adsorbate.


In [ ]:
# Adsorption-screen Toolkit settings: surface chemistry starts here.
TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None  # open MACE default used for the runnable tutorial path

# Geometry-optimization controls. TOOLKIT_DT is the FIRE2 optimizer step size,
# not a molecular-dynamics time step. TOOLKIT_FMAX is the stopping criterion:
# the largest active-atom force must fall below this value in eV/A.
TOOLKIT_DT = 0.01
TOOLKIT_N_STEPS = 1000  # hard safety cap; converged structures stop earlier once TOOLKIT_FMAX is reached
TOOLKIT_FMAX = 0.05     # eV/A, a screening-level force threshold for batched MLIP relaxations

# D3(BJ) is available in Toolkit workflows. This reference-backed tutorial keeps
# keep it disabled because the OC20Dense comparison data used below follows the
# non-D3 OC20 convention. To enable D3 for another application, replace None
# with a ToolkitD3BJConfig(...) or an equivalent config dict and record it in metadata.
TOOLKIT_D3BJ = None

# Adsorption structures are relaxed in batches. The H2O sweep and adsorption
# batch-size sweep above shows how batch size depends on chemistry, model, and VRAM.
BATCH_SIZE = {"short": 4, "full": 12}[RUN_SCOPE]
TOOLKIT_MODEL_LABEL = f"{TOOLKIT_CHECKPOINT} (head={TOOLKIT_HEAD})" if TOOLKIT_HEAD else TOOLKIT_CHECKPOINT

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

if TOOLKIT_D3BJ is None:
    print(f"Adsorption screen uses {TOOLKIT_MODEL_LABEL} with D3(BJ) disabled.")
else:
    print(f"Adsorption screen uses {TOOLKIT_MODEL_LABEL} with D3(BJ) enabled.")

RELAXATION_CONFIG = ToolkitRelaxationConfig(
    name="toolkit",
    cache_dir=CACHE_DIR,
    use_cached_responses=USE_SAVED_TUTORIAL_RESULTS,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_head=TOOLKIT_HEAD,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_D3BJ is not None,
)

try:
    RELAXATION_ENGINE = get_toolkit_relaxation_engine(RELAXATION_CONFIG)
except Exception as exc:
    raise RuntimeError(
        "Toolkit adsorption model construction failed. This usually means the "
        "selected checkpoint/head or acceleration flags are not supported by the "
        "current Toolkit runtime. First retry with TOOLKIT_ENABLE_CUEQ = False; "
        "if that still fails, retry with TOOLKIT_COMPILE_MODEL = False. "
        f"Original error: {type(exc).__name__}: {exc}"
    ) from exc

print(f"Toolkit relaxation engine ready    : {RELAXATION_ENGINE.name}")
print(f"TOOLKIT_CHECKPOINT                 : {TOOLKIT_CHECKPOINT}")
print(f"TOOLKIT_HEAD                       : {TOOLKIT_HEAD or 'not set'}")
print(f"TOOLKIT_N_STEPS                    : {TOOLKIT_N_STEPS}")
print(f"TOOLKIT_FMAX                       : {TOOLKIT_FMAX} eV/A")
print(f"TOOLKIT_D3BJ                       : {'enabled' if TOOLKIT_D3BJ else 'disabled'}")
print(f"BATCH_SIZE                         : {BATCH_SIZE}")
print(
    "Acceleration path                 : "
    f"compile_model={TOOLKIT_COMPILE_MODEL}, enable_cueq={TOOLKIT_ENABLE_CUEQ}"
)


In [ ]:
import json

# Reference helpers: build gas molecules and relax clean slabs/gas references in Toolkit batches.
import asyncio


def gas_reference_atoms(name: str) -> ase.Atoms:
    atoms = ADSORBATE_BUILDERS[name](ADSORBATE_ORIENTATIONS_FOR_SCREEN[name][0])
    atoms.set_cell([15.0, 15.0, 15.0])
    atoms.set_pbc(True)
    atoms.center()
    return atoms


async def relax_payload_batch(label: str, payloads: list):
    return await RELAXATION_ENGINE.async_relax(payloads, label=label, cellopt=False, session=None)


async def relax_reference_structures():
    clean_payloads = [
        ase_to_atomic_data(
            HOSTS[name],
            structure_id=f"clean_{safe_artifact_label(name)}",
            active_mask=make_active_mask(HOSTS[name], bottom_fraction=0.5),
        )
        for name in HOST_NAMES
    ]
    gas_payloads = [
        ase_to_atomic_data(gas_reference_atoms(name), structure_id=f"gas_{name}")
        for name in ADSORBATES
    ]

    clean_replies = []
    for start in range(0, len(clean_payloads), min(BATCH_SIZE, 3)):
        clean_replies.extend((await relax_payload_batch(
            f"surface_screen_clean_refs_{start // min(BATCH_SIZE, 3) + 1:02d}",
            clean_payloads[start:start + min(BATCH_SIZE, 3)],
        )).atoms)

    gas_reply = await relax_payload_batch("surface_screen_gas_refs", gas_payloads)
    return clean_replies, gas_reply.atoms


# SURFACE_SCREEN_OUTPUT_ROOT is resolved once in the control/housekeeping cells.
SURFACE_SCREEN_PATHS = surface_screen_output_paths(SURFACE_SCREEN_OUTPUT_ROOT)



In [ ]:
# Load or run clean-slab and gas-reference relaxations.
from ase.io import read as ase_read

if USE_SAVED_TUTORIAL_RESULTS and not REFRESH_SAVED_RESULTS:
    clean_df_cached = pd.read_csv(require_surface_screen_artifact(SURFACE_SCREEN_PATHS, "clean_slab_energies_csv", relpath=tutorial_relpath))
    gas_df_cached = pd.read_csv(require_surface_screen_artifact(SURFACE_SCREEN_PATHS, "gas_energies_csv", relpath=tutorial_relpath))

    E_HOST, HOST_RELAXED = {}, {}
    for _, row in clean_df_cached.iterrows():
        host = row["host"]
        E_HOST[host] = float(row["energy_eV"])
        HOST_RELAXED[host] = ase_read(row["structure_path"])

    E_ADS_GAS = {
        row["adsorbate"]: float(row["energy_eV"])
        for _, row in gas_df_cached.iterrows()
    }
    clean_opts_for_artifacts = []
    gas_opts_for_artifacts = []
    gas_relaxed_for_artifacts = {}
    print("Loaded saved clean-slab and gas-reference relaxations:")
    clean_display_cols = [c for c in ["host", "converged", "optimizer_nsteps", "max_force_eV_A", "energy_eV"] if c in clean_df_cached]
    gas_display_cols = [c for c in ["adsorbate", "converged", "optimizer_nsteps", "max_force_eV_A", "energy_eV"] if c in gas_df_cached]
    display(clean_df_cached[clean_display_cols])
    display(gas_df_cached[gas_display_cols])
else:
    clean_opts, gas_opts = await relax_reference_structures()

    E_HOST, HOST_RELAXED = {}, {}
    clean_rows = []
    for name, opt in zip(HOST_NAMES, clean_opts):
        relaxed = atomic_data_to_ase(opt)
        HOST_RELAXED[name] = relaxed
        E_HOST[name] = float(opt.energy)
        fmax = float(np.max(np.linalg.norm(np.array(opt.forces).reshape(-1, 3), axis=1)))
        clean_rows.append({
            "surface": name,
            "atoms": len(relaxed),
            "converged": opt.converged,
            "steps": opt.num_optimization_steps,
            "max |F| (eV/A)": round(fmax, 4),
            "E_clean_slab (eV)": round(E_HOST[name], 4),
        })

    E_ADS_GAS = {}
    GAS_RELAXED = {}
    gas_rows = []
    for name, opt in zip(ADSORBATES, gas_opts):
        relaxed = atomic_data_to_ase(opt)
        GAS_RELAXED[name] = relaxed
        E_ADS_GAS[name] = float(opt.energy)
        gas_rows.append({
            "adsorbate": name,
            "converged": opt.converged,
            "steps": opt.num_optimization_steps,
            "E_gas (eV)": round(E_ADS_GAS[name], 4),
        })

    clean_opts_for_artifacts = clean_opts
    gas_opts_for_artifacts = gas_opts
    gas_relaxed_for_artifacts = GAS_RELAXED

    display(pd.DataFrame(clean_rows))
    display(pd.DataFrame(gas_rows))

# Rebuild site candidates from the clean-slab structures selected above.
# In compute mode these are the freshly relaxed slabs; in saved-results mode
# they are the saved clean-slab structures.
SURFACE_SITE_CANDIDATES_BY_HOST = {
    host: SURFACE_SITE_FINDERS[host](HOST_RELAXED[host])
    for host in HOST_NAMES
}
missing_site_classes = []
for host, site_names in SURFACE_SITES_BY_HOST.items():
    available = SURFACE_SITE_CANDIDATES_BY_HOST[host]
    missing_site_classes.extend(
        (host, site) for site in site_names if site not in available or not available[site]
    )
if missing_site_classes:
    raise RuntimeError(f"Site finder did not produce requested site classes: {missing_site_classes}")
print("Site candidates are tied to the active clean-slab geometries.")
site_candidate_count_df = pd.DataFrame([
    {
        "surface": host,
        "candidate positions per class": ", ".join(
            f"{site}:{len(SURFACE_SITE_CANDIDATES_BY_HOST[host][site])}"
            for site in SURFACE_SITES_BY_HOST[host]
        ),
    }
    for host in HOST_NAMES
])
display(site_candidate_count_df)


---

## Run the batched adsorption panel

The full panel has 36 adsorbate/surface pairs. Each pair has six starts, so the adsorption part contains 216 independent geometry optimizations. With a batch size of 12, the work is packed as 18 Toolkit adsorption batches: two adsorbate/surface questions per GPU batch.



In [ ]:
# Build the visible configuration grid for the selected run scope.
ALL_PAIRS = [(host, adsorbate) for host in HOST_NAMES for adsorbate in ADSORBATES]
TOTAL_AVAILABLE_PAIRS = len(ALL_PAIRS)

RUN_PROFILES = {
    "short": {
        "pairs": [("Cu(111)", "CO")],
        "description": "one adsorbate/surface pair / six starts",
    },
    "full": {
        "pairs": ALL_PAIRS,
        "description": "9 surfaces x 4 adsorbates x 6 starts",
    },
}

RUN_PROFILE = RUN_PROFILES[RUN_SCOPE]
PAIRS = list(RUN_PROFILE["pairs"])

GRID: dict[tuple[str, str], list[Configuration]] = {}
grid_progress = make_notebook_progress(
    title="Starting-geometry grid",
    total=len(PAIRS),
    unit="pairs",
    message="ready to build grids",
    width_px=560,
)
for index, (host, adsorbate) in enumerate(PAIRS, start=1):
    grid_progress.update(done=index - 1, message=f"building {adsorbate} on {host}")
    GRID[(host, adsorbate)] = build_adsorption_starting_grid(host=host, adsorbate=adsorbate)
    grid_progress.update(done=index, message=f"finished {adsorbate} on {host}")

TOTAL_STARTING_GEOMETRIES = sum(len(configs) for configs in GRID.values())
print(f"Running {RUN_SCOPE_LABEL}: {RUN_PROFILE['description']}.")
print(f"Adsorbate/surface pairs: {len(PAIRS)} of {TOTAL_AVAILABLE_PAIRS}")
print(f"Starting geometries     : {TOTAL_STARTING_GEOMETRIES}")
print(f"Toolkit batch size      : {BATCH_SIZE}")
for (host, adsorbate), configs in GRID.items():
    print(f"  {adsorbate:<5} on {host:<9}: {len(configs)} starts")


In [ ]:
# Audit starting geometries before relaxation: no buried adsorbates, no accidental collisions.
all_configs = [config for configs in GRID.values() for config in configs]
slab_atom_counts = {host: len(HOST_RELAXED[host]) for host in HOST_NAMES}
initial_geometry_audit = audit_initial_configs(all_configs, slab_atom_counts)
problem_starts = initial_geometry_audit[initial_geometry_audit["audit_status"].ne("ok")]
if len(problem_starts):
    display(problem_starts[["pair", "label", "min_adsorbate_slab_distance_A", "audit_status"]])
    raise RuntimeError("Initial geometry audit found starts that need review before relaxation.")

print(f"Initial geometry audit passed for {len(initial_geometry_audit)} starts.")
display(
    initial_geometry_audit
    .groupby(["host", "adsorbate"], as_index=False)
    .agg(starts=("label", "count"), min_distance_A=("min_adsorbate_slab_distance_A", "min"), atoms=("n_atoms_total", "max"))
    .head(12)
)


### Visualize starting configurations

Inspect a few starts before relaxation. This is not decoration: it is the manual check that the programmatic site and orientation rules are doing what the text says they do.


In [ ]:
sample = []
for (host, adsorbate), configs in GRID.items():
    if configs:
        sample.append((f"{adsorbate}/{host}: {configs[0].site}, {configs[0].orientation}", configs[0].atoms))
    if len(sample) >= 9:
        break
display_widgets_row(sample, width="240px", height="215px", show_cell=True)


## Relax every starting configuration in Toolkit batches

The scientific unit is an adsorbate/surface question with six starts. The execution unit is a Toolkit batch, which can hold starts from more than one question. Batching accelerates independent structures without changing the chemical definition of each search.


In [ ]:
# Pack six-start scientific questions into Toolkit batches of up to BATCH_SIZE structures.
def packed_adsorption_batches():
    batches = []
    current = []
    for host, adsorbate in PAIRS:
        group = [(host, adsorbate, config) for config in GRID[(host, adsorbate)]]
        if current and len(current) + len(group) > BATCH_SIZE:
            batches.append(current)
            current = []
        current.extend(group)
    if current:
        batches.append(current)
    return batches


async def relax_all_adsorption_batches():
    batches = packed_adsorption_batches()
    out_results: dict[tuple[str, str], list[OptimizationResult]] = {pair: [] for pair in PAIRS}
    progress = make_notebook_progress(
        title="Adsorption relaxation",
        total=TOTAL_STARTING_GEOMETRIES,
        unit="structures",
        message=f"ready: {len(batches)} Toolkit batches, up to {BATCH_SIZE} structures each",
        average_label="s/structure",
        width_px=680,
    )
    done = 0
    for batch_index, batch_items in enumerate(batches, start=1):
        pairs_in_batch = sorted({f"{ads}/{host}" for host, ads, _ in batch_items})
        progress.update(
            done=done,
            message=f"running batch {batch_index}/{len(batches)}: {len(batch_items)} structures | {'; '.join(pairs_in_batch[:2])}",
        )
        data_list = [
            ase_to_atomic_data(config.atoms, structure_id=config.label, active_mask=config.active_mask)
            for _, _, config in batch_items
        ]
        reply = await RELAXATION_ENGINE.async_relax(
            data_list,
            label=f"surface_screen_adsorption_batch_{batch_index:02d}",
            cellopt=False,
            session=None,
        )
        for (host, adsorbate, _), result in zip(batch_items, reply.atoms):
            out_results[(host, adsorbate)].append(result)
        done += len(batch_items)
        progress.update(done=done, message=f"finished batch {batch_index}/{len(batches)}")
    return {
        pair: RelaxationBatchResult(
            atoms=[result.model_dump() if hasattr(result, "model_dump") else result for result in results],
            status="Success",
            info=f"toolkit batched adsorption screen; pair={pair[1]}/{pair[0]}",
        )
        for pair, results in out_results.items()
    }


if USE_SAVED_TUTORIAL_RESULTS and not REFRESH_SAVED_RESULTS:
    require_surface_screen_artifact(SURFACE_SCREEN_PATHS, "adsorption_results_csv", relpath=tutorial_relpath)
    pair_replies = {}
    missing_raw = []
    for pair in PAIRS:
        results = []
        for config in GRID[pair]:
            path = surface_screen_result_json_path(SURFACE_SCREEN_PATHS, config.label)
            if not path.exists():
                missing_raw.append(path)
                continue
            results.append(OptimizationResult.model_validate(json.loads(path.read_text(encoding="utf-8"))))
        pair_replies[pair] = RelaxationBatchResult(
            atoms=results,
            status="Success",
            info=f"loaded saved surface-screen results; pair={pair[1]}/{pair[0]}",
        )
    if missing_raw:
        raise RuntimeError(
            "Saved surface-screen result files are missing. First missing file: "
            f"{tutorial_relpath(missing_raw[0])}"
        )
    print(f"Loaded saved surface-screen relaxations for {len(pair_replies)} pair(s).")
else:
    pair_replies = await relax_all_adsorption_batches()
    os.makedirs(SURFACE_SCREEN_PATHS["raw"], exist_ok=True)
    os.makedirs(SURFACE_SCREEN_PATHS["initial_structures"], exist_ok=True)
    os.makedirs(SURFACE_SCREEN_PATHS["relaxed_structures"], exist_ok=True)
    from ase.io import write as ase_write

    for pair, reply in pair_replies.items():
        for config, result in zip(GRID[pair], reply.atoms):
            raw_path = surface_screen_result_json_path(SURFACE_SCREEN_PATHS, config.label)
            raw_path.write_text(result.model_dump_json(indent=2), encoding="utf-8")
            ase_write(SURFACE_SCREEN_PATHS["initial_structures"] / f"{safe_artifact_label(config.label)}.extxyz", config.atoms)
            ase_write(SURFACE_SCREEN_PATHS["relaxed_structures"] / f"{safe_artifact_label(config.label)}.extxyz", atomic_data_to_ase(result))

print(f"Relaxed {sum(len(r.atoms) for r in pair_replies.values())} configurations across {len(pair_replies)} pair(s).")


## Rank the batched screen by adsorption energy

For each relaxed structure, compute the same adsorption-energy expression used in the clean-slab and gas-reference cells:

`E_ads = E(slab + adsorbate) - E(clean slab) - E(gas molecule)`

The heatmap keeps the screen readable: each tile is the best relaxed adsorption energy found among the six starts for that adsorbate/surface pair. The number inside the tile is the rank of that surface for the same adsorbate.


In [ ]:
# Convert relaxed structures into adsorption-energy tables using the visible reference convention.
print(ADSORPTION_ENERGY_FORMULA)
# Do not reuse the H2O gas energy from the molecule-only batching demo here.
# Adsorption energies use gas references computed with the same surface-chemistry model.

# If a live exploratory branch reaches this cell without a gas reference, compute it
# with the same visible adsorbate builder and first listed orientation used above.
needed_gas_refs = [
    ads_name
    for ads_name in sorted({ads for _, ads in PAIRS})
    if ads_name not in E_ADS_GAS
]
gas_progress = None
if needed_gas_refs:
    gas_progress = make_notebook_progress(
        title="Gas references",
        total=len(needed_gas_refs),
        unit="molecules",
        message="ready to relax gas references",
        width_px=560,
    )
for index, ads_name in enumerate(needed_gas_refs, start=1):
    if gas_progress is not None:
        gas_progress.update(done=index - 1, message=f"relaxing {ads_name}")
    mol = ADSORBATE_BUILDERS[ads_name](ADSORBATE_ORIENTATIONS_FOR_SCREEN[ads_name][0])
    mol.set_cell([15, 15, 15])
    mol.set_pbc(True)
    mol.center()
    r = RELAXATION_ENGINE.relax(
        [ase_to_atomic_data(mol, structure_id=f"gas_{ads_name}")],
        label=f"gas_{ads_name}",
    )
    E_ADS_GAS[ads_name] = float(r.atoms[0].energy)
    print(f"E({ads_name}, gas) = {E_ADS_GAS[ads_name]:.4f} eV")
    if gas_progress is not None:
        gas_progress.update(done=index, message=f"finished {ads_name}")

PAIR_RESULTS: dict[tuple[str, str], pd.DataFrame] = {}
for (host, adsorbate), reply in pair_replies.items():
    PAIR_RESULTS[(host, adsorbate)] = build_pair_results_table(
        host=host,
        adsorbate=adsorbate,
        configs=GRID[(host, adsorbate)],
        opt_results=reply.atoms,
        clean_slab_atoms=HOST_RELAXED[host],
        e_clean_slab_ev=E_HOST[host],
        e_gas_ads_ev=E_ADS_GAS[adsorbate],
        execution_path=EXECUTION_PATH,
        reliability_max_force_eV_A=RELIABILITY_MAX_FORCE_EV_A,
        desorption_height_A=DESORPTION_HEIGHT_A,
    )

# More negative E_ads means stronger binding. Avoid a bare "max E" label here:
# the numerical maximum is the weakest/least negative adsorption energy.
pair_energy_rows = pd.concat(
    [df.assign(pair=f"{a}/{h}") for (h, a), df in PAIR_RESULTS.items()],
    ignore_index=True,
)
summary_table = (
    pair_energy_rows
    .groupby("pair", as_index=False)
    .agg(
        strongest_binding_E_ads_eV=("E_ads (eV)", "min"),   # most negative
        median_E_ads_eV=("E_ads (eV)", "median"),
        weakest_binding_E_ads_eV=("E_ads (eV)", "max"),     # least negative
        n_relaxed_starts=("E_ads (eV)", "count"),
    )
    .sort_values("strongest_binding_E_ads_eV")
    .round(3)
)
summary_table


### Adsorption-energy heatmap

This is the main result of the worked screen: which surface in the panel gives the lowest-energy relaxed structure for each molecular probe.


In [ ]:
# Build the screen-level tables used by the heatmap and final-structure widgets.
adsorption_results_df = pd.concat(
    [df.assign(host=host, adsorbate=adsorbate) for (host, adsorbate), df in PAIR_RESULTS.items()],
    ignore_index=True,
)
batch_summary_df = pd.DataFrame([
    {
        "batch_label": f"surface_screen_adsorption_batch_{index:02d}",
        "batch_type": "adsorption",
        "n_structures": len(batch),
        "loaded_from_cache": bool(USE_SAVED_TUTORIAL_RESULTS and not REFRESH_SAVED_RESULTS),
    }
    for index, batch in enumerate(packed_adsorption_batches(), start=1)
])
step_statistics_df = build_step_statistics(
    adsorption_results_df,
    green_step_max=200,
    yellow_step_max=500,
    force_threshold_eV_A=TOOLKIT_FMAX,
)
surface_pair_summary_df = summarize_surface_screen_pairs(
    adsorption_results_df,
    step_statistics_df,
    exclude_red_step_status=True,
)
application_heatmap_df = build_application_heatmap(
    surface_pair_summary_df,
    adsorbate_hints={spec.name: spec.application_hint for spec in SURFACE_SCREEN_ADSORBATES},
)
difficult_cases_df = build_difficult_cases(adsorption_results_df, step_statistics_df)
ranked_winners_df = (
    surface_pair_summary_df
    .sort_values(["adsorbate", "rank_within_adsorbate", "best_E_ads_eV"])
    .groupby("adsorbate", as_index=False)
    .head(1)
    .sort_values("adsorbate")
    .reset_index(drop=True)
)


In [ ]:
# Plot the heatmap and show the best surface found for each adsorbate.
heatmap_path = os.path.join(PLOTS_DIR, "surface_screen_adsorption_heatmap.png")
if USE_SAVED_TUTORIAL_RESULTS and not REFRESH_SAVED_RESULTS:
    if os.path.exists(heatmap_path):
        display_inline(heatmap_path)
        print(f"Using saved figure: {heatmap_path}")
    else:
        session_heatmap_path = os.path.join(
            LIVE_OUTPUT_DIR, "tutorial", "plots", "surface_screen_adsorption_heatmap.png"
        )
        session_heatmap_path = plot_surface_screen_heatmap(
            application_heatmap_df,
            output_path=session_heatmap_path,
            title="Best adsorption energy found by the batched screen",
        )
        display_inline(session_heatmap_path)
        print(f"Saved session figure: {session_heatmap_path}")
else:
    heatmap_path = plot_surface_screen_heatmap(
        application_heatmap_df,
        output_path=heatmap_path,
        title="Best adsorption energy found by the batched screen",
    )
    display_inline(heatmap_path)
    print(f"Saved: {heatmap_path}")

display(
    ranked_winners_df[[
        "adsorbate", "host", "best_E_ads_eV", "best_final_site",
        "n_starting_geometries", "best_optimizer_nsteps", "best_step_status",
    ]]
    .rename(columns={
        "host": "best surface in this panel",
        "best_E_ads_eV": "best E_ads (eV)",
        "best_final_site": "final site",
        "n_starting_geometries": "starts searched",
        "best_optimizer_nsteps": "relaxation steps",
        "best_step_status": "step status",
    })
    .style.hide(axis="index")
)


## Inspect the ranked final structures

The heatmap ranks energies; the structures below make the ranking inspectable. For each molecule, show the relaxed structure from the lowest-energy surface in this panel.


In [ ]:
# Show the top-ranked relaxed structure for each adsorbate as interactive OVITO widgets.
import importlib

importlib.invalidate_caches()
from helpers.surface_screen_widgets import display_surface_screen_winner_widgets

winner_widget_result = display_surface_screen_winner_widgets(
    surface_pair_summary_df=globals().get("surface_pair_summary_df"),
    surface_screen_paths=globals().get("SURFACE_SCREEN_PATHS"),
    columns=2,
    width="460px",
    height="360px",
    header_height="92px",
    show_cell=True,
    wrap_periodic_cell=False,
)

ranked_winners_df = winner_widget_result["ranked_winners_df"]
ranked_winner_items = winner_widget_result["items"]
winner_pair_summary_path = winner_widget_result["pair_summary_path"]
winner_artifact_root = winner_widget_result["artifact_root"]


---

## Interpreting the adsorption screen

The heatmap is a screening result for this worked panel: after generating a local adsorption search space across surfaces, adsorbates, sites, orientations, and starting geometries, which relaxed structures bind most strongly with the selected MACE-MPA model and reference convention?

For chemistry research, this opens many possibilities to continue meaningful work where the process of interest starts when an adsorbate attaches to the surface: catalysis, mixture separation, protective film deposition, and water harvesting.

Batching changes the practical modus operandi. Instead of asking one structure for one answer, the workflow generates many plausible structures, relaxes them on the GPU, and ranks the final geometries so the researcher can see a large space all at once. This notebook only gives a glimpse of that pattern; the important point is that screening becomes an executable stage of the workflow rather than a manual preselection step.

<p style="margin:16px 0 4px; max-width:100%;">
  <img src="assets/images/adsorption_slab_1x_vs_20x_black.png" alt="Batching accelerates discovery: one slab equals 1x, batched slabs equal 20x" style="display:block; max-width:100%; height:auto; background:#000; border:1px solid #111; border-radius:8px;">
</p>

## Some limitations of this tutorial - and how model/data selection can address them

This notebook focuses on throughput screening for adsorption configurations. The limits below define which downstream questions need additional methods.

- **Activation barriers** are not computed. Thermodynamic binding is not a kinetic pathway; NEB, dimer, or string methods are required for transition states. Coverage and multi-molecule effects require separate simulations.
- **Open-shell, magnetic 3d, reducible-oxide defect, and f-electron chemistry** are excluded from these examples - MACE-MPA-0 cannot describe them. MLIPs that go beyond ground state properties could be useful for future discovery. 
- **Dispersion conventions matter.** This notebook disables D3 to match OC20Dense/OC20 reference data; in general, enabling D3 is a great inexpensive way to improve accuracy of intermolecular interactions and surface chemistry. 

## Let's stay in touch

Deepest appreciation to everyone involved, especially Wen Jie Ong, Ryan Reese, Roman Zubatyuk, Sepideh Khajehei, and the OpenHackathon team for support, feedback, and collaboration around this tutorial.

For questions, feedback, follow-up discussion, or ideas for extending the workflow, reach out to [Nikita Fedik](mailto:nfedik@nvidia.com) or [Justin Smith](mailto:jusmith@nvidia.com). We would love to hear from you and discuss how **<span style="color:#76b900; font-weight:600;">NVIDIA ALCHEMI</span>** could help accelerate your workflow.

🔗 **ALCHEMI resources:** [ALCHEMI hub](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) · [Toolkit GitHub](https://github.com/NVIDIA/nvalchemi-toolkit) · [Toolkit-Ops GitHub](https://github.com/NVIDIA/nvalchemi-toolkit-ops)

📰 **ALCHEMI blogs:** [ALCHEMI discovery blog](https://developer.nvidia.com/blog/revolutionizing-ai-driven-material-discovery-using-nvidia-alchemi/) · [Toolkit workflows blog](https://developer.nvidia.com/blog/building-custom-atomistic-simulation-workflows-for-chemistry-and-materials-science-with-nvidia-alchemi-toolkit/) · [Toolkit-Ops blog](https://developer.nvidia.com/blog/accelerating-ai-powered-chemistry-and-materials-science-simulations-with-nvidia-alchemi-toolkit-ops/)